# Mark 1 → Mark 4E — Unified Research Pipeline (Single Notebook Reproduction)

[![Phase](https://img.shields.io/badge/Research%20Phase-Mark%201%20to%204E-blue.svg)]()
[![Status](https://img.shields.io/badge/Mark%204E-Controlling%20Pass-success.svg)]()

## What this notebook is

This single notebook consolidates the **eight sequential research notebooks** executed in
`mark 1/` (`mark_1_probability_contrast_localization_diagnostic.ipynb` →
`mark_2_roi_multiwindow_feasibility.ipynb` → `mark_3_two_stage_multiwindow_overfit.ipynb` →
`mark_4_two_stage_validation_smoke.ipynb` → `mark_4b_roi_probability_diagnostics.ipynb` →
`mark_4c_two_channel_recall_ablation.ipynb` → `mark_4d_metric_reconciliation_v116_diagnostic.ipynb`
→ `mark_4e_checkpoint_fusion_validation.ipynb`) into **one runnable, structured pipeline**.

It reproduces **the same outputs and results** (gate JSONs, metrics, and visualizations) and writes
them with the same naming convention into:

```
Evaluation/mark_1_to_4e_outputs/
├── mark_1_outputs/   … mark_1_gate_result.json, calibration_*, probability_localization_*, hu_contrast_*
├── mark_2_outputs/   … mark_2_gate_result.json, roi_*, multiwindow_*, selected_roi_*
├── mark_3_outputs/   … mark_3_gate_result.json, training_roi_*, overfit_*, roundtrip_*
├── mark_4_outputs/   … mark_4_gate_result.json, mark_4_history.csv, best_validation_*, broad_png_*
├── mark_4b_outputs/  … mark_4b_gate_result.json, threshold_results.csv, localization_volume_*
├── mark_4c_outputs/  … mark_4c_gate_result.json, arm_comparison.csv, ablation_learning_dashboard.png
├── mark_4d_outputs/  … mark_4d_gate_result.json, reconciled_*, v116_*, positive_patient_heatmap.png
├── mark_4e_outputs/  … mark_4e_gate_result.json, fusion_*, selected_fusion_*
└── consolidated/     … unified_gate_summary.csv, reproduction_verification.json
```

## Table of contents

| # | Section | Original notebook | Key result |
|---|---|---|---|
| 0 | Global setup | — | provenance, test lock, shared helpers |
| 1 | Mark 1 diagnostic | `mark_1_probability_contrast_localization_diagnostic.ipynb` | calibration gate failed (signal suppressed) |
| 2 | Mark 2 ROI feasibility | `mark_2_roi_multiwindow_feasibility.ipynb` | ROI contains 100% tumor, 42.7% median crop |
| 3 | Mark 3 overfit gate | `mark_3_two_stage_multiwindow_overfit.ipynb` | broad-1ch overfit Dice 0.9006 |
| 4 | Mark 4 validation smoke | `mark_4_two_stage_validation_smoke.ipynb` | 5/6 continuation targets |
| 5 | Mark 4B calibration | `mark_4b_roi_probability_diagnostics.ipynb` | threshold alone cannot fix recall |
| 6 | Mark 4C ablation | `mark_4c_two_channel_recall_ablation.ipynb` | neither arm passed the full gate |
| 7 | Mark 4D reconciliation | `mark_4d_metric_reconciliation_v116_diagnostic.ipynb` | V116 is a localization failure, not ROI clipping |
| 8 | Mark 4E fusion | `mark_4e_checkpoint_fusion_validation.ipynb` | **maximum fusion @ 0.70 passes all 6 targets** |

## Execution modes

- **REUSE mode (default, fast):** loads the frozen probability caches, ROI manifests, history CSVs and
  checkpoints from `mark 1/`, then recomputes every metric, gate and figure from those artifacts.
  Deterministic — the gate JSONs reproduce the originals exactly.
- **REBUILD mode:** set the `REBUILD_*`/`RUN_*` flags below to `True` to re-run validation inference
  (rebuild `.npz` caches) and/or retrain the models from scratch (GPU hours).

**The test split stays locked in both modes.**

> **Run order:** `Kernel → Restart Kernel and Run All`. The notebook is fully self-contained.

## Global setup

Everything below mirrors the shared preamble of the eight original notebooks:
paths, seeds, hashes, targets, and the framework modules (`VerifiedManifestDataset`,
`MobileNetV2UNet`, `FocalDiceLoss`) are verified before any computation starts.

In [ ]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
OUT = {name: OUTPUT_ROOT / f"{name}_outputs" for name in
       ["mark_1", "mark_2", "mark_3", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
CONSOLIDATED = OUTPUT_ROOT / "consolidated"
for _d in list(OUT.values()) + [CONSOLIDATED]:
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {OUTPUT_ROOT}")

### How to read the sections

Every Mark section follows the same pattern:

1. **Context** — what question the mark answers and the frozen contract (gates/targets).
2. **Data** — verify provenance, reuse or rebuild caches/artifacts.
3. **Results** — recompute the metrics/gates and save CSVs, JSONs and figures.
4. **Gate** — write the machine-readable `mark_X_gate_result.json` into `Evaluation/mark_1_to_4e_outputs/`.
5. **Reproduction check** — compare the recomputed gate against the original `mark 1/` gate.

Section 9 consolidates every gate into one table and writes the verification report.

# Part 1 — Mark 1: Probability, Contrast & Localization Diagnostic

**Original:** `mark 1/mark_1_probability_contrast_localization_diagnostic.ipynb`

## Question

Does the frozen epoch-8 multi-task checkpoint already localize tumour signal somewhere in the
probability map, and can global calibration (threshold + predicted-liver support) recover it?

## Key finding (reproduced)

- **Calibration gate: FAILED.** No global configuration passed every guardrail.
- Best observed configuration (raw mode, tumor threshold 0.70): mean patient Dice ≈ 0.333, V104 ≈ 0,
  V116 ≈ 0 — the signal is present but **mis-localized / suppressed**, not merely under-thresholded.
- Classification: `mislocalized_or_absent_signal` → next experiment = predicted-liver ROI.

## Contract

- Validation-only. No training. No patient-specific thresholds. Test split locked.
- Guardrail targets (Mark 1): mean patient Dice ≥ 0.406915, V104 ≥ 0.50, V116 ≥ 0.05,
  Q1 detection ≥ 45%, positive predicted-empty ≤ 20%, empty-slice FP ≤ 15%.

### 1.1 Load the epoch-8 checkpoint strictly and verify geometry

In [ ]:
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

checkpoint = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 8
assert checkpoint["manifest_sha256"] == EXPECTED_MANIFEST_SHA256

model = MobileNetV2UNet(in_channels=1, out_channels=2, pretrained=False)
model.load_state_dict(checkpoint["model_state"], strict=True)
model.to(DEVICE).eval()

with torch.inference_mode():
    probe = model(torch.zeros(1, 1, 256, 256, device=DEVICE))
    probe_prob = torch.sigmoid(probe)
assert tuple(probe.shape) == (1, 2, 256, 256)
assert torch.isfinite(probe_prob).all()
print("PASS: strict epoch-8 checkpoint load;"
      f" checkpoint={source_checkpoint_hash[:12]}...")

### 1.2 Reproduce validation preprocessing and build the validation loader

In [ ]:
class Mark1ValidationDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.rows = base_dataset.rows

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        sample = self.base[index]
        image = image_robust_normalize(sample["image"][0].numpy())
        with Image.open(self.rows[index]["organ_mask_path"]) as handle:
            organ = (np.asarray(handle.convert("L"), dtype=np.uint8) > 0).astype(np.uint8)
        return {
            "image": torch.from_numpy(image[None]).float(),
            "tumor_mask": sample["mask"].to(torch.uint8),
            "organ_mask": torch.from_numpy(organ[None]),
            "sample_id": sample["sample_id"],
            "volume_id": int(sample["volume_id"]),
            "slice_index": int(sample["slice_index"]),
        }

validation_base = VerifiedManifestDataset(
    MANIFEST_PATH, split="val", root_dir=DATASET_ROOT,
    target="tumor", transform=None, validate_paths=True,
)
validation_dataset = Mark1ValidationDataset(validation_base)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
assert len(validation_dataset) == EXPECTED_VALIDATION_SLICES if False else True
assert len(validation_dataset) == 10_685
print(f"READY: validation loader contains {len(validation_dataset):,} slices.")

### 1.3 Cache full-image probabilities per volume (reuse frozen cache or rebuild)

In [ ]:
ORIG_CACHE = MARK1_DIR / "mark_1_outputs" / "probability_cache"
CACHE_DIR = OUT["mark_1"] / "probability_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def save_volume_cache(volume_id, bucket, target_dir):
    order = np.argsort(bucket["slice_index"])
    payload = {key: np.asarray(value)[order] for key, value in bucket.items()}
    np.savez_compressed(target_dir / f"volume_{volume_id}.npz", **payload)


def build_probability_cache():
    current_volume, bucket, processed = None, None, []
    model.eval()
    with torch.inference_mode():
        for batch in validation_loader:
            probabilities = torch.sigmoid(
                model(batch["image"].to(DEVICE, non_blocking=True))).cpu().numpy()
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                if current_volume is None or volume_id != current_volume:
                    if current_volume is not None:
                        save_volume_cache(current_volume, bucket, CACHE_DIR)
                        processed.append(current_volume)
                    current_volume = volume_id
                    bucket = {"sample_id": [], "slice_index": [],
                              "tumor_truth": [], "organ_truth": [],
                              "liver_probability": [], "tumor_probability": []}
                bucket["sample_id"].append(str(sample_id))
                bucket["slice_index"].append(int(batch["slice_index"][index]))
                bucket["tumor_truth"].append(batch["tumor_mask"][index, 0].numpy().astype(np.uint8))
                bucket["organ_truth"].append(batch["organ_mask"][index, 0].numpy().astype(np.uint8))
                bucket["liver_probability"].append(probabilities[index, 0].astype(np.float16))
                bucket["tumor_probability"].append(probabilities[index, 1].astype(np.float16))
    if current_volume is not None:
        save_volume_cache(current_volume, bucket, CACHE_DIR)
        processed.append(current_volume)
    return processed


existing = sorted(CACHE_DIR.glob("volume_*.npz"))
if REUSE_CACHES and len(existing) == 13:
    print(f"REUSE: {len(existing)} patient caches already present in {CACHE_DIR}.")
elif REUSE_CACHES and len(list(ORIG_CACHE.glob("volume_*.npz"))) == 13:
    import shutil
    for p in ORIG_CACHE.glob("volume_*.npz"):
        shutil.copy2(p, CACHE_DIR / p.name)
    print("REUSE: copied 13 frozen patient caches from mark_1_outputs/probability_cache.")
else:
    processed = build_probability_cache()
    print(f"REBUILD: cached {len(processed)} volumes.")

m1_cache = {}
coverage = []
for path in sorted(CACHE_DIR.glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        item = {key: payload[key] for key in payload.files}
    expected = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)].sort_values("slice_index")
    assert len(item["sample_id"]) == len(expected)
    assert item["sample_id"].astype(str).tolist() == expected["sample_id"].astype(str).tolist()
    assert np.array_equal(item["slice_index"], expected["slice_index"].to_numpy())
    for key in ("tumor_truth", "organ_truth", "liver_probability", "tumor_probability"):
        assert item[key].shape[1:] == (256, 256)
    for key in ("liver_probability", "tumor_probability"):
        assert np.isfinite(item[key]).all()
        assert float(item[key].min()) >= 0 and float(item[key].max()) <= 1
    m1_cache[volume_id] = item
    coverage.append({"volume_id": volume_id, "slices": len(item["sample_id"]),
                     "tumor_positive_slices": int(item["tumor_truth"].any(axis=(1, 2)).sum()),
                     "cache_dtype": str(item["tumor_probability"].dtype),
                     "cache_mb": path.stat().st_size / (1024 ** 2)})
coverage_frame = pd.DataFrame(coverage)
assert len(m1_cache) == 13 and int(coverage_frame["slices"].sum()) == 10_685
coverage_frame.to_csv(OUT["mark_1"] / "cache_coverage.csv", index=False)
display(coverage_frame)
print("PASS: complete, ordered, finite validation cache.")

### 1.4 Slice probability statistics and localization panels (V104, V116)

In [ ]:
def probability_statistics(cached):
    rows = []
    for volume_id, item in cached.items():
        for index, sample_id in enumerate(item["sample_id"]):
            truth = item["tumor_truth"][index].astype(bool)
            organ = item["organ_truth"][index].astype(bool)
            probability = item["tumor_probability"][index].astype(np.float32)
            true_values = probability[truth]
            liver_background = probability[organ & ~truth]
            extra_liver = probability[~organ]
            rows.append({
                "sample_id": str(sample_id), "volume_id": volume_id,
                "slice_index": int(item["slice_index"][index]),
                "true_pixels": int(truth.sum()),
                "max_tumor_probability": float(probability.max()),
                "mean_probability_inside_truth": float(true_values.mean()) if true_values.size else np.nan,
                "max_probability_inside_truth": float(true_values.max()) if true_values.size else np.nan,
                "mean_liver_background_probability": float(liver_background.mean()) if liver_background.size else np.nan,
                "mean_extra_liver_probability": float(extra_liver.mean()) if extra_liver.size else np.nan,
            })
    return pd.DataFrame(rows)


def localization_panel(volume_id, statistics):
    item = m1_cache[volume_id]
    candidates = statistics.loc[
        statistics["volume_id"].eq(volume_id) & statistics["true_pixels"].gt(0)].copy()
    selected = list(dict.fromkeys([
        candidates["true_pixels"].idxmax(),
        (candidates["true_pixels"] - candidates["true_pixels"].median()).abs().idxmin(),
        candidates["true_pixels"].idxmin(),
        candidates["mean_probability_inside_truth"].idxmax(),
        candidates["mean_probability_inside_truth"].idxmin(),
    ]))
    figure, axes = plt.subplots(len(selected), 5, figsize=(18, 3.6 * len(selected)))
    if len(selected) == 1:
        axes = axes[None, :]
    for row_axes, row_index in zip(axes, selected):
        row = statistics.loc[row_index]
        index = int(np.where(item["slice_index"] == row["slice_index"])[0][0])
        manifest_row = validation_manifest.loc[
            validation_manifest["sample_id"].eq(row["sample_id"])].iloc[0]
        with Image.open(DATASET_ROOT / manifest_row["image_path"]) as handle:
            image = image_robust_normalize(
                np.asarray(handle.convert("L"), dtype=np.float32) / 255.0)
        truth = item["tumor_truth"][index].astype(bool)
        liver_probability = item["liver_probability"][index].astype(np.float32)
        tumor_probability = item["tumor_probability"][index].astype(np.float32)
        prediction = tumor_probability >= 0.50
        error = np.zeros((*truth.shape, 3), dtype=np.float32)
        error[truth & ~prediction, 0] = 1.0
        error[prediction & ~truth, 2] = 1.0
        panels = [
            (image, "Normalized CT", "gray", None),
            (image, "Truth contour", "gray", truth),
            (liver_probability, "Liver probability", "viridis", None),
            (tumor_probability, "Tumor probability", "magma", None),
            (error, "FN red / FP blue", None, None),
        ]
        for axis, (panel, title, cmap, contour) in zip(row_axes, panels):
            if panel.ndim == 2:
                axis.imshow(panel, cmap=cmap, vmin=0, vmax=1)
            else:
                axis.imshow(panel)
            if contour is not None and contour.any():
                axis.contour(contour, levels=[0.5], colors=["#00FFFF"], linewidths=1)
            axis.set_title(title)
            axis.axis("off")
        row_axes[0].set_ylabel(
            f"{row['sample_id']}\ntrue={row['true_pixels']:,}\npmax={row['max_tumor_probability']:.3f}",
            fontsize=8)
    figure.suptitle(f"Volume {volume_id} probability localization", fontsize=16, weight="bold")
    figure.tight_layout()
    path = OUT["mark_1"] / f"probability_localization_{volume_id}.png"
    figure.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()


m1_slice_stats = probability_statistics(m1_cache)
m1_slice_stats.to_csv(OUT["mark_1"] / "probability_slice_statistics.csv", index=False)
for focus_volume in (104, 116):
    localization_panel(focus_volume, m1_slice_stats)
print("PASS: slice statistics + localization panels written.")

### 1.5 Fixed-bin probability populations

In [ ]:
focus_volumes = [104, 116, 108, 109, 110]
bins = np.linspace(0, 1, 51)
figure, axes = plt.subplots(len(focus_volumes), 1, figsize=(12, 3.2 * len(focus_volumes)))
rng = np.random.default_rng(SEED)
for axis, volume_id in zip(axes, focus_volumes):
    item = m1_cache[volume_id]
    probability = item["tumor_probability"].astype(np.float32)
    truth = item["tumor_truth"].astype(bool)
    organ = item["organ_truth"].astype(bool)
    populations = {
        "True tumor": probability[truth],
        "Liver background": probability[organ & ~truth],
        "Extra-liver": probability[~organ],
    }
    for label, values in populations.items():
        if values.size > 500_000:
            values = rng.choice(values, 500_000, replace=False)
        axis.hist(values, bins=bins, density=True, histtype="step",
                  linewidth=1.5, label=f"{label} (n={len(values):,})")
    axis.set_yscale("log")
    axis.set_xlim(0, 1)
    axis.set_title(f"Volume {volume_id}")
    axis.set_xlabel("Tumor probability")
    axis.set_ylabel("Density (log)")
    axis.legend(fontsize=8)
figure.suptitle("Tumor-probability populations", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_1"] / "probability_population_histograms.png",
               dpi=170, bbox_inches="tight")
plt.show()

### 1.6 Source-NIfTI HU contrast (tumor vs liver)

In [ ]:
import nibabel as nib


def robust_effect_size(tumor_values, liver_values):
    tumor_mad = np.median(np.abs(tumor_values - np.median(tumor_values)))
    liver_mad = np.median(np.abs(liver_values - np.median(liver_values)))
    pooled = max(1.4826 * np.sqrt((tumor_mad ** 2 + liver_mad ** 2) / 2), 1e-6)
    return float((np.median(tumor_values) - np.median(liver_values)) / pooled)


def compute_hu_contrast():
    rows = []
    positive_rows = validation_manifest.loc[validation_manifest["tumor_pixels"].gt(0)]
    for volume_id, group in positive_rows.groupby("volume_id", sort=True):
        first = group.iloc[0]
        ct_image = nib.load(str(first["source_volume_path"]))
        segmentation = nib.load(str(first["source_segmentation_path"]))
        transform = str(first["transform_applied"])
        for row in group.sort_values("slice_index").itertuples(index=False):
            z = int(row.slice_index)
            hu = np.asanyarray(ct_image.dataobj[:, :, z]).astype(np.float32)
            labels = np.asanyarray(segmentation.dataobj[:, :, z]).astype(np.uint8)
            if transform == "rot180":
                labels = np.rot90(labels, 2).copy()
            elif transform != "identity":
                raise ValueError(f"Unsupported transform: {transform}")
            tumor_values = hu[labels == 2]
            liver_values = hu[labels == 1]
            if not tumor_values.size or not liver_values.size:
                continue
            rows.append({
                "sample_id": row.sample_id, "volume_id": int(volume_id),
                "slice_index": z,
                "tumor_pixels_native": int(tumor_values.size),
                "liver_background_pixels_native": int(liver_values.size),
                "tumor_mean_hu": float(tumor_values.mean()),
                "tumor_median_hu": float(np.median(tumor_values)),
                "liver_background_mean_hu": float(liver_values.mean()),
                "liver_background_median_hu": float(np.median(liver_values)),
                "mean_contrast_hu": float(tumor_values.mean() - liver_values.mean()),
                "median_contrast_hu": float(np.median(tumor_values) - np.median(liver_values)),
                "robust_effect_size": robust_effect_size(tumor_values, liver_values),
            })
    return pd.DataFrame(rows)


hu_slice = compute_hu_contrast()
assert not hu_slice.empty
hu_slice.to_csv(OUT["mark_1"] / "hu_contrast_per_slice.csv", index=False)
hu_volume = hu_slice.groupby("volume_id").agg(
    positive_slices=("sample_id", "size"),
    median_contrast_hu=("median_contrast_hu", "median"),
    mean_contrast_hu=("mean_contrast_hu", "mean"),
    median_effect_size=("robust_effect_size", "median")).reset_index()
hu_volume.to_csv(OUT["mark_1"] / "hu_contrast_per_volume.csv", index=False)
display(hu_volume)

### 1.7 Sweep global tumor thresholds and predicted-liver support

In [ ]:
COARSE_TUMOR_THRESHOLDS = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
                                   dtype=np.float32)
LIVER_THRESHOLDS = np.array([0.30, 0.40, 0.50, 0.60, 0.70], dtype=np.float32)
LIVER_DILATION_KERNELS = [1, 5, 11, 21, 31]


def dilated_support(probability, threshold, kernel):
    binary = torch.from_numpy((probability >= threshold).astype(np.float32))[:, None]
    if kernel > 1:
        binary = F.max_pool2d(binary, kernel_size=kernel, stride=1, padding=kernel // 2)
    return binary[:, 0].numpy() > 0


def evaluate_configuration(tumor_threshold, liver_threshold=None, kernel=1):
    patient_rows, slice_rows = [], []
    total_intersection = total_predicted = total_true = 0
    positive_empty = positive_count = empty_fp = empty_count = 0
    removed_pixels = removed_true = removed_false = 0
    for volume_id, item in m1_cache.items():
        truth = item["tumor_truth"].astype(bool)
        raw = item["tumor_probability"].astype(np.float32) >= tumor_threshold
        if liver_threshold is None:
            prediction = raw
        else:
            support = dilated_support(item["liver_probability"].astype(np.float32),
                                      liver_threshold, kernel)
            prediction = raw & support
            removed = raw & ~prediction
            removed_pixels += int(removed.sum())
            removed_true += int((removed & truth).sum())
            removed_false += int((removed & ~truth).sum())
        intersection = prediction & truth
        predicted_pixels = prediction.sum(axis=(1, 2))
        true_pixels = truth.sum(axis=(1, 2))
        intersections = intersection.sum(axis=(1, 2))
        total_intersection += int(intersections.sum())
        total_predicted += int(predicted_pixels.sum())
        total_true += int(true_pixels.sum())
        positive = true_pixels > 0
        empty = ~positive
        positive_count += int(positive.sum())
        positive_empty += int((positive & (predicted_pixels == 0)).sum())
        empty_count += int(empty.sum())
        empty_fp += int((empty & (predicted_pixels > 0)).sum())
        patient_rows.append({
            "volume_id": volume_id, "true_pixels": int(true_pixels.sum()),
            "predicted_pixels": int(predicted_pixels.sum()),
            "intersection_pixels": int(intersections.sum()),
            "micro_dice": float((2 * intersections.sum() + 1e-6)
                                / (predicted_pixels.sum() + true_pixels.sum() + 1e-6)),
        })
        for index in range(len(true_pixels)):
            slice_rows.append({
                "sample_id": str(item["sample_id"][index]), "volume_id": volume_id,
                "true_pixels": int(true_pixels[index]),
                "predicted_pixels": int(predicted_pixels[index]),
                "intersection_pixels": int(intersections[index]),
            })
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = pd.qcut(
        positive_slices["true_pixels"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
    q1 = positive_slices.loc[positive_slices["size_quartile"].eq("Q1")]
    return {
        "tumor_threshold": float(tumor_threshold),
        "liver_threshold": float(liver_threshold) if liver_threshold is not None else np.nan,
        "dilation_kernel": int(kernel),
        "mode": "raw" if liver_threshold is None else "liver_supported",
        "global_dice": (2 * total_intersection + 1e-6) / (total_predicted + total_true + 1e-6),
        "pixel_precision": total_intersection / max(total_predicted, 1),
        "pixel_recall": total_intersection / max(total_true, 1),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_positive_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": 100 * float((q1["intersection_pixels"] > 0).mean()),
        "positive_predicted_empty_pct": 100 * positive_empty / max(positive_count, 1),
        "empty_slice_false_positive_pct": 100 * empty_fp / max(empty_count, 1),
        "predicted_tumor_pixels": total_predicted,
        "pixels_removed_by_liver_support": removed_pixels,
        "true_pixels_removed_by_liver_support": removed_true,
        "false_pixels_removed_by_liver_support": removed_false,
    }, patients


m1_config_rows, m1_patient_rows = [], []
for tumor_threshold in COARSE_TUMOR_THRESHOLDS:
    result, patients = evaluate_configuration(float(tumor_threshold))
    m1_config_rows.append(result)
    m1_patient_rows.append(patients.assign(tumor_threshold=float(tumor_threshold),
                                           liver_threshold=np.nan, dilation_kernel=1, mode="raw"))
    for liver_threshold in LIVER_THRESHOLDS:
        for kernel in LIVER_DILATION_KERNELS:
            result, patients = evaluate_configuration(float(tumor_threshold),
                                                      float(liver_threshold), kernel)
            m1_config_rows.append(result)
            m1_patient_rows.append(patients.assign(
                tumor_threshold=float(tumor_threshold), liver_threshold=float(liver_threshold),
                dilation_kernel=kernel, mode="liver_supported"))

m1_config_results = pd.DataFrame(m1_config_rows)
m1_patient_config = pd.concat(m1_patient_rows, ignore_index=True)
m1_config_results.to_csv(OUT["mark_1"] / "calibration_configuration_results.csv", index=False)
m1_patient_config.to_csv(OUT["mark_1"] / "calibration_patient_metrics.csv", index=False)
display(m1_config_results.sort_values(
    ["mean_patient_dice", "empty_slice_false_positive_pct"],
    ascending=[False, True]).head(15))
print(f"Evaluated {len(m1_config_results)} configurations (234 expected).")

### 1.8 Preregistered acceptance gate + calibration frontier

In [ ]:
def gate_flags(frame):
    return (
        frame["mean_patient_dice"].ge(FINAL_TARGETS["mean_patient_dice"])
        & frame["volume_104_dice"].ge(FINAL_TARGETS["volume_104_dice"])
        & frame["volume_116_dice"].ge(FINAL_TARGETS["volume_116_dice"])
        & frame["q1_detected_pct"].ge(FINAL_TARGETS["q1_detected_pct"])
        & frame["positive_predicted_empty_pct"].le(FINAL_TARGETS["positive_predicted_empty_pct"])
        & frame["empty_slice_false_positive_pct"].le(FINAL_TARGETS["empty_slice_false_positive_pct"])
    )


m1_config_results["all_targets_passed"] = gate_flags(m1_config_results)
eligible = m1_config_results.loc[m1_config_results["all_targets_passed"]].copy()
if not eligible.empty:
    m1_selected_config = eligible.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]
else:
    m1_selected_config = m1_config_results.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]

figure, axes = plt.subplots(2, 3, figsize=(18, 10))
raw = m1_config_results.loc[m1_config_results["mode"].eq("raw")]
axes[0, 0].plot(raw["tumor_threshold"], raw["mean_patient_dice"], marker="o")
axes[0, 0].axhline(FINAL_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 0].set_title("Mean patient Dice")
axes[0, 1].plot(raw["tumor_threshold"], raw["volume_104_dice"], marker="o", label="V104")
axes[0, 1].plot(raw["tumor_threshold"], raw["volume_116_dice"], marker="s", label="V116")
axes[0, 1].set_title("Focus-patient Dice"); axes[0, 1].legend()
axes[0, 2].plot(raw["tumor_threshold"], raw["q1_detected_pct"], marker="o")
axes[0, 2].axhline(FINAL_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[0, 2].set_title("Q1 detection (%)")
axes[1, 0].plot(raw["empty_slice_false_positive_pct"], raw["mean_patient_dice"], marker="o")
axes[1, 0].axvline(FINAL_TARGETS["empty_slice_false_positive_pct"], linestyle="--", color="#444")
axes[1, 0].set_title("Patient Dice vs empty-slice FP"); axes[1, 0].set_xlabel("Empty-slice FP (%)")
axes[1, 1].plot(raw["positive_predicted_empty_pct"], raw["q1_detected_pct"], marker="o")
axes[1, 1].set_title("Q1 detection vs positive-empty")
axes[1, 1].set_xlabel("Positive predicted empty (%)")
removal = m1_config_results.loc[m1_config_results["mode"].eq("liver_supported")]
axes[1, 2].hist(removal["pixels_removed_by_liver_support"], bins=30, color="#2878B5")
axes[1, 2].set_title("Pixels removed by liver support")
for axis in axes.flat:
    axis.grid(alpha=0.25)
figure.suptitle("Mark 1 calibration frontier", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_1"] / "calibration_frontier_dashboard.png",
               dpi=170, bbox_inches="tight")
plt.show()
print("Selected diagnostic configuration:", m1_selected_config.to_dict())

### 1.9 Bootstrap patient uncertainty + write the Mark 1 gate

In [ ]:
selected_filter = (
    m1_patient_config["mode"].eq(m1_selected_config["mode"])
    & m1_patient_config["tumor_threshold"].eq(m1_selected_config["tumor_threshold"])
    & m1_patient_config["dilation_kernel"].eq(m1_selected_config["dilation_kernel"])
)
if m1_selected_config["mode"] == "liver_supported":
    selected_filter &= m1_patient_config["liver_threshold"].eq(m1_selected_config["liver_threshold"])
selected_patients = m1_patient_config.loc[selected_filter].copy()
selected_positive = selected_patients.loc[
    selected_patients["true_pixels"].gt(0), "micro_dice"].to_numpy()
rng = np.random.default_rng(SEED)
bootstrap_means = np.array([
    rng.choice(selected_positive, size=len(selected_positive), replace=True).mean()
    for _ in range(1_000)
])
m1_bootstrap = pd.DataFrame([{
    "configuration_mode": m1_selected_config["mode"],
    "tumor_threshold": m1_selected_config["tumor_threshold"],
    "liver_threshold": m1_selected_config["liver_threshold"],
    "dilation_kernel": m1_selected_config["dilation_kernel"],
    "patients": len(selected_positive), "bootstrap_iterations": 1_000,
    "mean_dice_p2_5": np.percentile(bootstrap_means, 2.5),
    "mean_dice_p50": np.percentile(bootstrap_means, 50),
    "mean_dice_p97_5": np.percentile(bootstrap_means, 97.5),
    "original_median": np.median(selected_positive),
    "original_q25": np.percentile(selected_positive, 25),
    "original_q75": np.percentile(selected_positive, 75),
}])
m1_bootstrap.to_csv(OUT["mark_1"] / "bootstrap_confidence_intervals.csv", index=False)
display(m1_bootstrap)

calibration_passed = bool(m1_config_results["all_targets_passed"].any())
selected = m1_selected_config.to_dict()
if calibration_passed:
    failure_category, next_mark = "calibration_success", "freeze_global_configuration"
else:
    focus_stats = m1_slice_stats.loc[
        m1_slice_stats["volume_id"].isin([104, 116]) & m1_slice_stats["true_pixels"].gt(0)]
    localized_probability = float(focus_stats["max_probability_inside_truth"].median())
    if localized_probability >= 0.10:
        failure_category, next_mark = ("weak_but_localized",
                                       "stable_recall_objective_or_predicted_liver_normalization")
    elif hu_volume is not None and (
            hu_volume.loc[hu_volume["volume_id"].isin([104, 116]),
                          "median_effect_size"].abs().median() < 0.5):
        failure_category, next_mark = ("low_source_contrast",
                                       "controlled_multi_window_source_nifti_experiment")
    else:
        failure_category, next_mark = ("mislocalized_or_absent_signal",
                                       "predicted_liver_roi_or_capacity_experiment")

m1_gate = {
    "status": "mark_1_diagnostic_complete",
    "calibration_gate_passed": calibration_passed,
    "failure_category": failure_category,
    "selected_global_configuration": selected if calibration_passed else None,
    "best_observed_configuration_for_diagnosis": selected,
    "manifest_sha256": manifest_hash,
    "checkpoint_sha256": source_checkpoint_hash,
    "parent_checkpoint_sha256": checkpoint["source_checkpoint_sha256"],
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "test_images_accessed": False,
    "next_mark": next_mark,
    "manual_review_required": not calibration_passed,
}
(OUT["mark_1"] / "mark_1_gate_result.json").write_text(json.dumps(m1_gate, indent=2))
display(pd.DataFrame([m1_gate]).T.rename(columns={0: "value"}))

# ---- Reproduction check against the original gate ----
orig_m1 = json.loads((MARK1_DIR / "mark_1_outputs" / "mark_1_gate_result.json").read_text())
_check = orig_m1["best_observed_configuration_for_diagnosis"]
recomputed = m1_gate["best_observed_configuration_for_diagnosis"]
diffs = {k: abs(float(recomputed[k]) - float(_check[k])) for k in
         ["global_dice", "mean_patient_dice", "volume_104_dice", "volume_116_dice",
          "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"]}
print("Mark 1 reproduction check (|recomputed - original|):", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 1 gate drifted from the original!"
print("PASS: Mark 1 gate matches the original mark_1_gate_result.json.")

# Part 2 — Mark 2: Predicted-Liver ROI & Multi-Window Feasibility

**Original:** `mark 1/mark_2_roi_multiwindow_feasibility.ipynb`

## Question

Before another tumor model is trained: can one **global, prediction-only** 3D liver ROI contain every
validation tumor (including V104/V116), and do fixed source-HU windows improve tumor/liver separation?

## Key finding (reproduced)

- **ROI gate: PASSED.** liver threshold 0.50, padding 16, `largest_3d` component:
  V104 and V116 tumor containment = 1.0, minimum positive-patient/slice containment = 1.0,
  zero empty patient ROIs, **median crop-area ratio 0.427** (~58% slice-search reduction).
- Fixed windows `broad [-160,240]`, `liver [-20,140]`, `narrow [20,120]` are diagnostic inputs
  for Mark 3, not yet proven model improvements.

## Contract

- ROI uses **predicted-liver probabilities only** — ground truth is never an ROI input.
- Feasibility gate: V104 ≥ 99%, V116 ≥ 99%, min positive-patient containment ≥ 99%,
  min positive-slice containment ≥ 99%, zero empty ROIs, median crop ratio ≤ 60%.
- Test split locked.

### 2.1 Load the Mark 1 caches and verify the Mark 1 gate

In [ ]:
import shutil

mark1_gate = json.loads((OUT["mark_1"] / "mark_1_gate_result.json").read_text())
assert mark1_gate["status"] == "mark_1_diagnostic_complete"
assert mark1_gate["calibration_gate_passed"] is False
assert mark1_gate["test_images_accessed"] is False

m2_cache = {}
for path in sorted((OUT["mark_1"] / "probability_cache").glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        m2_cache[volume_id] = {key: payload[key] for key in payload.files}
assert len(m2_cache) == 13
cache_coverage = pd.DataFrame([
    {"volume_id": v, "slices": len(i["sample_id"]),
     "tumor_pixels": int(i["tumor_truth"].sum()),
     "tumor_positive_slices": int(i["tumor_truth"].any(axis=(1, 2)).sum())}
    for v, i in m2_cache.items()])
cache_coverage.to_csv(OUT["mark_2"] / "cache_coverage.csv", index=False)
display(cache_coverage)
print("PASS: 13 complete, ordered probability caches reused from Mark 1.")

### 2.2 Construct prediction-only 3D ROIs and score containment

In [ ]:
from scipy import ndimage

ROI_LIVER_THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50]
PADDINGS = [16, 32, 48, 64]
COMPONENT_MODES = ["all", "largest_3d"]


def predicted_liver_mask(probability, threshold, component_mode):
    mask = probability.astype(np.float32) >= threshold
    if component_mode == "all" or not mask.any():
        return mask
    labels, count = ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))
    if count == 0:
        return mask
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def bbox_from_mask(mask, padding=0):
    if not mask.any():
        return None
    _, ys, xs = np.where(mask)
    return (max(int(ys.min()) - padding, 0), min(int(ys.max()) + 1 + padding, 256),
            max(int(xs.min()) - padding, 0), min(int(xs.max()) + 1 + padding, 256))


def score_roi(volume_id, item, threshold, padding, component_mode, base_box):
    box = bbox_from_mask(base_box, padding)
    truth = item["tumor_truth"].astype(bool)
    positive_slices = truth.any(axis=(1, 2))
    if box is None:
        contained = np.zeros_like(truth)
        area_ratio = 0.0
    else:
        y0, y1, x0, x1 = box
        contained = np.zeros_like(truth)
        contained[:, y0:y1, x0:x1] = truth[:, y0:y1, x0:x1]
        area_ratio = ((y1 - y0) * (x1 - x0)) / (256 * 256)
    contained_pixels = int(contained.sum())
    truth_pixels = int(truth.sum())
    contained_positive_slices = (contained.any(axis=(1, 2)) & positive_slices).sum()
    return {
        "volume_id": volume_id, "liver_threshold": threshold, "padding": padding,
        "component_mode": component_mode, "roi_empty": box is None,
        "y0": box[0] if box else np.nan, "y1": box[1] if box else np.nan,
        "x0": box[2] if box else np.nan, "x1": box[3] if box else np.nan,
        "crop_area_ratio": area_ratio, "tumor_pixels": truth_pixels,
        "tumor_pixel_containment": (contained_pixels / truth_pixels if truth_pixels else np.nan),
        "positive_slices": int(positive_slices.sum()),
        "positive_slice_containment": (contained_positive_slices / positive_slices.sum()
                                       if positive_slices.any() else np.nan),
    }


roi_patient_rows = []
for threshold in ROI_LIVER_THRESHOLDS:
    for component_mode in COMPONENT_MODES:
        for volume_id, item in m2_cache.items():
            liver_mask = predicted_liver_mask(item["liver_probability"], threshold, component_mode)
            base_box = bbox_from_mask(liver_mask)
            for padding in PADDINGS:
                roi_patient_rows.append(score_roi(volume_id, item, threshold, padding,
                                                  component_mode, base_box))
roi_patient_results = pd.DataFrame(roi_patient_rows)
roi_patient_results.to_csv(OUT["mark_2"] / "roi_patient_results.csv", index=False)
print(f"Evaluated {len(roi_patient_results):,} patient/configuration rows.")

### 2.3 Aggregate configurations and apply the feasibility gate

In [ ]:
configuration_rows = []
group_columns = ["liver_threshold", "padding", "component_mode"]
for keys, group in roi_patient_results.groupby(group_columns, sort=True):
    threshold, padding, component_mode = keys
    positive = group.loc[group["tumor_pixels"].gt(0)]
    indexed = group.set_index("volume_id")
    configuration_rows.append({
        "liver_threshold": threshold, "padding": padding,
        "component_mode": component_mode,
        "volume_104_tumor_containment": indexed.at[104, "tumor_pixel_containment"],
        "volume_116_tumor_containment": indexed.at[116, "tumor_pixel_containment"],
        "minimum_positive_patient_containment": positive["tumor_pixel_containment"].min(),
        "mean_positive_patient_containment": positive["tumor_pixel_containment"].mean(),
        "minimum_positive_slice_containment": positive["positive_slice_containment"].min(),
        "median_crop_area_ratio": group["crop_area_ratio"].median(),
        "maximum_crop_area_ratio": group["crop_area_ratio"].max(),
        "empty_patient_rois": int(group["roi_empty"].sum()),
    })

roi_configurations = pd.DataFrame(configuration_rows)
roi_configurations["hard_containment_gate_passed"] = (
    roi_configurations["volume_104_tumor_containment"].ge(0.99)
    & roi_configurations["volume_116_tumor_containment"].ge(0.99)
    & roi_configurations["minimum_positive_patient_containment"].ge(0.99)
    & roi_configurations["minimum_positive_slice_containment"].ge(0.99)
    & roi_configurations["empty_patient_rois"].eq(0))
roi_configurations["efficient_roi_gate_passed"] = (
    roi_configurations["hard_containment_gate_passed"]
    & roi_configurations["median_crop_area_ratio"].le(0.60))
roi_configurations.to_csv(OUT["mark_2"] / "roi_configuration_results.csv", index=False)

eligible = roi_configurations.loc[roi_configurations["efficient_roi_gate_passed"]].copy()
fallback = roi_configurations.loc[roi_configurations["hard_containment_gate_passed"]].copy()
candidate_pool = eligible if not eligible.empty else fallback
if not candidate_pool.empty:
    selected_roi = candidate_pool.sort_values(
        ["median_crop_area_ratio", "minimum_positive_patient_containment"],
        ascending=[True, False]).iloc[0]
else:
    selected_roi = roi_configurations.sort_values(
        ["minimum_positive_patient_containment", "minimum_positive_slice_containment",
         "median_crop_area_ratio"], ascending=[False, False, True]).iloc[0]

display(roi_configurations.sort_values(
    ["hard_containment_gate_passed", "minimum_positive_patient_containment",
     "median_crop_area_ratio"], ascending=[False, False, True]).head(20))
print("Selected diagnostic ROI:", selected_roi.to_dict())

### 2.4 ROI feasibility dashboards

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(17, 12))
for mode, marker in [("all", "o"), ("largest_3d", "s")]:
    subset = roi_configurations.loc[roi_configurations["component_mode"].eq(mode)]
    axes[0, 0].scatter(subset["median_crop_area_ratio"],
                       subset["minimum_positive_patient_containment"],
                       label=mode, marker=marker, s=60, alpha=0.8)
axes[0, 0].axhline(0.99, linestyle="--", color="#444444")
axes[0, 0].axvline(0.60, linestyle=":", color="#444444")
axes[0, 0].set_xlabel("Median crop-area ratio")
axes[0, 0].set_ylabel("Minimum positive-patient containment")
axes[0, 0].set_title("Containment versus crop burden"); axes[0, 0].legend()

pivot_104 = roi_configurations.loc[
    roi_configurations["component_mode"].eq("all")].pivot(
    index="liver_threshold", columns="padding", values="volume_104_tumor_containment")
image = axes[0, 1].imshow(pivot_104, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axes[0, 1].set_xticks(range(len(pivot_104.columns)), pivot_104.columns)
axes[0, 1].set_yticks(range(len(pivot_104.index)), pivot_104.index)
axes[0, 1].set_xlabel("Padding"); axes[0, 1].set_ylabel("Liver threshold")
axes[0, 1].set_title("V104 containment — all components")
figure.colorbar(image, ax=axes[0, 1], fraction=0.046)

pivot_116 = roi_configurations.loc[
    roi_configurations["component_mode"].eq("all")].pivot(
    index="liver_threshold", columns="padding", values="volume_116_tumor_containment")
image = axes[1, 0].imshow(pivot_116, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axes[1, 0].set_xticks(range(len(pivot_116.columns)), pivot_116.columns)
axes[1, 0].set_yticks(range(len(pivot_116.index)), pivot_116.index)
axes[1, 0].set_xlabel("Padding"); axes[1, 0].set_ylabel("Liver threshold")
axes[1, 0].set_title("V116 containment — all components")
figure.colorbar(image, ax=axes[1, 0], fraction=0.046)

selected_filter = (
    roi_patient_results["liver_threshold"].eq(selected_roi["liver_threshold"])
    & roi_patient_results["padding"].eq(selected_roi["padding"])
    & roi_patient_results["component_mode"].eq(selected_roi["component_mode"]))
selected_patients = roi_patient_results.loc[selected_filter].sort_values("volume_id")
axes[1, 1].bar(selected_patients["volume_id"].astype(str),
               selected_patients["tumor_pixel_containment"].fillna(1.0),
               color=["#2878B5" if v >= 0.99 else "#F28E2B"
                      for v in selected_patients["tumor_pixel_containment"].fillna(1.0)])
axes[1, 1].axhline(0.99, linestyle="--", color="#444444")
axes[1, 1].set_ylim(0, 1.03)
axes[1, 1].set_title("Selected ROI tumor containment by patient")
axes[1, 1].set_xlabel("Volume"); axes[1, 1].set_ylabel("Containment")

figure.suptitle("Predicted-liver ROI feasibility", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_2"] / "roi_feasibility_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

### 2.5 Inspect selected predicted ROIs for V104 and V116

In [ ]:
def load_derived_image(sample_id):
    row = validation_manifest.loc[validation_manifest["sample_id"].eq(sample_id)].iloc[0]
    with Image.open(DATASET_ROOT / row["image_path"]) as handle:
        return np.asarray(handle.convert("L"), dtype=np.float32) / 255.0


def selected_box_for_volume(volume_id):
    row = roi_patient_results.loc[
        selected_filter & roi_patient_results["volume_id"].eq(volume_id)].iloc[0]
    if row["roi_empty"]:
        return None
    return tuple(int(row[key]) for key in ("y0", "y1", "x0", "x1"))


figure, axes = plt.subplots(2, 4, figsize=(17, 9))
for row_axes, volume_id in zip(axes, [104, 116]):
    item = m2_cache[volume_id]
    tumor_sizes = item["tumor_truth"].sum(axis=(1, 2))
    index = int(np.argmax(tumor_sizes))
    sample_id = str(item["sample_id"][index])
    image = load_derived_image(sample_id)
    liver_probability = item["liver_probability"][index].astype(np.float32)
    tumor = item["tumor_truth"][index].astype(bool)
    box = selected_box_for_volume(volume_id)

    row_axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
    row_axes[0].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    row_axes[0].set_title(f"V{volume_id} largest tumor slice")
    row_axes[1].imshow(liver_probability, cmap="viridis", vmin=0, vmax=1)
    row_axes[1].set_title("Predicted-liver probability")
    row_axes[2].imshow(image, cmap="gray", vmin=0, vmax=1)
    row_axes[2].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    if box:
        y0, y1, x0, x1 = box
        row_axes[2].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                            fill=False, edgecolor="#FF2D2D", linewidth=2))
    row_axes[2].set_title("Prediction-only 3D ROI")
    if box:
        y0, y1, x0, x1 = box
        row_axes[3].imshow(image[y0:y1, x0:x1], cmap="gray", vmin=0, vmax=1)
    else:
        row_axes[3].text(0.5, 0.5, "EMPTY ROI", ha="center", va="center")
    row_axes[3].set_title("ROI crop")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("Selected ROI inspection", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_2"] / "selected_roi_v104_v116.png", dpi=170, bbox_inches="tight")
plt.show()

### 2.6 Fixed-window source-HU visibility (train + validation)

In [ ]:
import nibabel as nib

WINDOWS = {"broad_abdominal": (-160.0, 240.0), "liver_soft_tissue": (-20.0, 140.0),
           "narrow_lesion": (20.0, 120.0)}


def transform_labels(labels, transform):
    if transform == "identity":
        return labels
    if transform == "rot180":
        return np.rot90(labels, 2).copy()
    raise ValueError(transform)


def window_statistics(split):
    rows = []
    split_manifest = manifest.loc[
        manifest["split"].eq(split) & manifest["tumor_pixels"].gt(0)]
    for volume_id, group in split_manifest.groupby("volume_id", sort=True):
        first = group.iloc[0]
        ct = nib.load(str(first["source_volume_path"]))
        segmentation = nib.load(str(first["source_segmentation_path"]))
        transform = str(first["transform_applied"])
        for row in group.sort_values("slice_index").itertuples(index=False):
            z = int(row.slice_index)
            hu = np.asanyarray(ct.dataobj[:, :, z]).astype(np.float32)
            labels = transform_labels(
                np.asanyarray(segmentation.dataobj[:, :, z]).astype(np.uint8), transform)
            tumor = hu[labels == 2]
            liver = hu[labels == 1]
            if not tumor.size or not liver.size:
                continue
            base = {"split": split, "sample_id": row.sample_id, "volume_id": int(volume_id),
                    "slice_index": z, "tumor_pixels_native": int(tumor.size),
                    "median_contrast_hu": float(np.median(tumor) - np.median(liver))}
            for name, (lower, upper) in WINDOWS.items():
                tumor_window = window_hu(tumor, (lower, upper))
                liver_window = window_hu(liver, (lower, upper))
                rows.append({
                    **base, "window": name, "lower_hu": lower, "upper_hu": upper,
                    "tumor_median_windowed": float(np.median(tumor_window)),
                    "liver_median_windowed": float(np.median(liver_window)),
                    "absolute_median_separation": float(
                        abs(np.median(tumor_window) - np.median(liver_window))),
                    "tumor_low_saturation_pct": float(100 * (tumor <= lower).mean()),
                    "tumor_high_saturation_pct": float(100 * (tumor >= upper).mean()),
                    "liver_low_saturation_pct": float(100 * (liver <= lower).mean()),
                    "liver_high_saturation_pct": float(100 * (liver >= upper).mean()),
                })
    return pd.DataFrame(rows)


window_slice_statistics = pd.concat(
    [window_statistics("train"), window_statistics("val")], ignore_index=True)
window_slice_statistics.to_csv(OUT["mark_2"] / "multiwindow_slice_statistics.csv", index=False)
window_summary = window_slice_statistics.groupby(["split", "window"]).agg(
    slices=("sample_id", "size"),
    median_absolute_separation=("absolute_median_separation", "median"),
    median_tumor_low_saturation_pct=("tumor_low_saturation_pct", "median"),
    median_tumor_high_saturation_pct=("tumor_high_saturation_pct", "median")).reset_index()
window_summary.to_csv(OUT["mark_2"] / "multiwindow_summary.csv", index=False)
display(window_summary)

### 2.7 Window separability dashboards + V104/V116 examples

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for name in WINDOWS:
    train_values = window_slice_statistics.loc[
        window_slice_statistics["split"].eq("train") & window_slice_statistics["window"].eq(name),
        "absolute_median_separation"]
    val_values = window_slice_statistics.loc[
        window_slice_statistics["split"].eq("val") & window_slice_statistics["window"].eq(name),
        "absolute_median_separation"]
    axes[0].hist(train_values, bins=40, density=True, histtype="step",
                 linewidth=1.5, label=f"{name} train")
    axes[0].hist(val_values, bins=40, density=True, histtype="step",
                 linewidth=1.5, linestyle="--", label=f"{name} val")
axes[0].set_title("Tumor–liver windowed separation")
axes[0].set_xlabel("Absolute median separation"); axes[0].set_ylabel("Density")
axes[0].legend(fontsize=7)

focus = window_slice_statistics.loc[
    window_slice_statistics["split"].eq("val")
    & window_slice_statistics["volume_id"].isin([104, 116])]
focus_box = [focus.loc[focus["volume_id"].eq(v) & focus["window"].eq(name),
                       "absolute_median_separation"].to_numpy()
             for v in (104, 116) for name in WINDOWS]
labels = [f"V{v}\n{name}" for v in (104, 116) for name in WINDOWS]
axes[1].boxplot(focus_box, tick_labels=labels, showfliers=False)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("V104/V116 separation by window")
axes[1].set_ylabel("Absolute median separation")

summary_pivot = window_summary.pivot(index="window", columns="split",
                                     values="median_absolute_separation")
summary_pivot.plot.bar(ax=axes[2], color=["#2878B5", "#F28E2B"])
axes[2].set_title("Median separation by split")
axes[2].set_ylabel("Absolute median separation")
axes[2].tick_params(axis="x", rotation=25)
axes[2].legend(title="Split")

figure.suptitle("Fixed source-HU window feasibility", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_2"] / "multiwindow_feasibility_dashboard.png",
               dpi=170, bbox_inches="tight")
plt.show()


def source_slice(volume_id, slice_index):
    row = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)
        & validation_manifest["slice_index"].eq(slice_index)].iloc[0]
    ct = nib.load(str(row["source_volume_path"]))
    segmentation = nib.load(str(row["source_segmentation_path"]))
    hu = np.asanyarray(ct.dataobj[:, :, slice_index]).astype(np.float32)
    labels = transform_labels(
        np.asanyarray(segmentation.dataobj[:, :, slice_index]).astype(np.uint8),
        str(row["transform_applied"]))
    return hu, labels


figure, axes = plt.subplots(2, len(WINDOWS) + 1, figsize=(18, 9))
for row_axes, volume_id in zip(axes, [104, 116]):
    item = m2_cache[volume_id]
    index = int(np.argmax(item["tumor_truth"].sum(axis=(1, 2))))
    slice_index = int(item["slice_index"][index])
    hu, labels = source_slice(volume_id, slice_index)
    tumor = labels == 2
    row_axes[0].imshow(hu, cmap="gray", vmin=-160, vmax=240)
    row_axes[0].contour(tumor, levels=[0.5], colors=["#00FFFF"])
    row_axes[0].set_title(f"V{volume_id} broad + truth")
    for axis, (name, (lower, upper)) in zip(row_axes[1:], WINDOWS.items()):
        axis.imshow(window_hu(hu, (lower, upper)), cmap="gray", vmin=0, vmax=1)
        axis.contour(tumor, levels=[0.5], colors=["#00FFFF"])
        axis.set_title(f"{name}\n[{lower:.0f}, {upper:.0f}] HU")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("V104 and V116 fixed-window inspection", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_2"] / "v104_v116_multiwindow_examples.png",
               dpi=170, bbox_inches="tight")
plt.show()

### 2.8 Write the Mark 2 feasibility gate

In [ ]:
roi_hard_passed = bool(selected_roi["hard_containment_gate_passed"])
roi_efficient_passed = bool(selected_roi["efficient_roi_gate_passed"])

focus_window_summary = window_slice_statistics.loc[
    window_slice_statistics["split"].eq("val")
    & window_slice_statistics["volume_id"].isin([104, 116])].groupby(
    ["volume_id", "window"]).agg(
    median_separation=("absolute_median_separation", "median"),
    tumor_low_saturation_pct=("tumor_low_saturation_pct", "median"),
    tumor_high_saturation_pct=("tumor_high_saturation_pct", "median")).reset_index()
focus_window_summary.to_csv(OUT["mark_2"] / "v104_v116_window_summary.csv", index=False)

if not roi_hard_passed:
    decision, next_notebook = "REVISE_LIVER_LOCALIZATION_OR_USE_SAFER_ANATOMY_ROI", "mark_2b_liver_roi_recovery"
elif roi_hard_passed and not roi_efficient_passed:
    decision, next_notebook = "ROI_CONTAINS_TUMOR_BUT_CROP_IS_TOO_BROAD", "mark_2b_roi_efficiency_ablation"
else:
    decision, next_notebook = "PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT", "mark_3_two_stage_multiwindow_overfit"

m2_gate = {
    "status": "mark_2_feasibility_complete",
    "roi_hard_containment_gate_passed": roi_hard_passed,
    "roi_efficiency_gate_passed": roi_efficient_passed,
    "selected_roi_configuration": {
        key: (selected_roi[key].item() if hasattr(selected_roi[key], "item") else selected_roi[key])
        for key in selected_roi.index},
    "fixed_windows": {key: list(value) for key, value in WINDOWS.items()},
    "decision": decision, "next_notebook": next_notebook,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "mark1_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "roi_uses_ground_truth": False, "test_images_accessed": False,
}
(OUT["mark_2"] / "mark_2_gate_result.json").write_text(json.dumps(m2_gate, indent=2))
display(pd.DataFrame([m2_gate]).T.rename(columns={0: "value"}))
display(focus_window_summary)
print("DECISION:", decision)

# ---- Reproduction check against the original gate ----
orig_m2 = json.loads((MARK1_DIR / "mark_2_outputs" / "mark_2_gate_result.json").read_text())
orig_sel = orig_m2["selected_roi_configuration"]
recomputed_sel = m2_gate["selected_roi_configuration"]
key_set = ["volume_104_tumor_containment", "volume_116_tumor_containment",
           "minimum_positive_patient_containment", "minimum_positive_slice_containment",
           "median_crop_area_ratio", "maximum_crop_area_ratio", "empty_patient_rois"]
diffs = {k: abs(float(recomputed_sel[k]) - float(orig_sel[k])) for k in key_set}
print("Mark 2 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 2 ROI gate drifted from the original!"
assert m2_gate["decision"] == orig_m2["decision"]
print("PASS: Mark 2 gate matches the original mark_2_gate_result.json.")

# Part 3 — Mark 3: Two-Stage Multi-Window Training-ROI & Overfit Gate

**Original:** `mark 1/mark_3_two_stage_multiwindow_overfit.ipynb`

## Question

Can the frozen pipeline (prediction-only liver ROI + source-HU windows) overfit a tiny stratified
cohort — proving the ROI geometry and input representation carry the signal — and which channel
configuration is the simplest one that passes?

## Key finding (reproduced)

- Training ROI gate: **PASSED** (100% tumor-pixel and positive-slice containment, 0 empty ROIs).
- Overfit gate: **PASSED** with `broad_1ch` — hard micro-Dice **0.9006** at epoch 17, 0% positive
  predicted-empty. The 2-channel and 3-channel arms were not simpler, so the 1-channel input won.

## Contract

- Training ROI rule: liver threshold 0.50, `largest_3d`, padding 16 (frozen from Mark 2).
- Overfit gate: hard micro-Dice ≥ 0.90, positive predicted-empty = 0%, finite loss/gradients,
  round-trip Dice ≥ 0.98, no test access.
- 16 slices (4 per size quartile) shared by all channel arms.

### 3.1 Verify Mark 2 authorization and load the frozen liver model

In [ ]:
mark2_gate = json.loads((OUT["mark_2"] / "mark_2_gate_result.json").read_text())
assert mark2_gate["status"] == "mark_2_feasibility_complete"
assert mark2_gate["roi_hard_containment_gate_passed"] is True
assert mark2_gate["roi_efficiency_gate_passed"] is True
assert mark2_gate["test_images_accessed"] is False
assert float(mark2_gate["selected_roi_configuration"]["liver_threshold"]) == 0.50
assert int(mark2_gate["selected_roi_configuration"]["padding"]) == 16
assert mark2_gate["selected_roi_configuration"]["component_mode"] == "largest_3d"

checkpoint = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 8
liver_model = MobileNetV2UNet(in_channels=1, out_channels=2, pretrained=False)
liver_model.load_state_dict(checkpoint["model_state"], strict=True)
liver_model.to(DEVICE).eval()
with torch.inference_mode():
    probe = liver_model(torch.zeros(1, 1, 256, 256, device=DEVICE))
assert probe.shape == (1, 2, 256, 256)
print("PASS: frozen liver model loaded; Mark 2 authorization confirmed.")

### 3.2 Freeze training ROIs (reuse frozen manifest or rebuild)

In [ ]:
import shutil
from scipy import ndimage

ORIG_ROI_MANIFEST = MARK1_DIR / "mark_3_outputs" / "training_roi_manifest.csv"
TRAIN_ROI_PATH = OUT["mark_3"] / "training_roi_manifest.csv"


def load_normalized_png(path):
    with Image.open(path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    return image_robust_normalize(image)


def largest_component(mask):
    labels, count = ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))
    if count == 0:
        return mask
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == sizes.argmax()


def padded_bbox(mask, padding):
    if not mask.any():
        return None
    _, ys, xs = np.where(mask)
    return (max(int(ys.min()) - padding, 0), min(int(ys.max()) + 1 + padding, 256),
            max(int(xs.min()) - padding, 0), min(int(xs.max()) + 1 + padding, 256))


def predict_volume_liver(group):
    probabilities = []
    paths = [DATASET_ROOT / path for path in group["image_path"]]
    with torch.inference_mode():
        for start in range(0, len(paths), 24):
            images = np.stack([load_normalized_png(path) for path in paths[start:start + 24]])
            batch = torch.from_numpy(images[:, None]).float().to(DEVICE)
            probabilities.append(torch.sigmoid(liver_model(batch))[:, 0].cpu().numpy())
    return np.concatenate(probabilities, axis=0)


def score_box(group, box):
    tumor_total = tumor_inside = positive_total = positive_inside = 0
    for row in group.itertuples(index=False):
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            tumor = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        tumor_pixels = int(tumor.sum())
        tumor_total += tumor_pixels
        if tumor_pixels:
            positive_total += 1
        if box is not None:
            y0, y1, x0, x1 = box
            inside = int(tumor[y0:y1, x0:x1].sum())
            tumor_inside += inside
            positive_inside += int(tumor_pixels > 0 and inside > 0)
    return {"tumor_pixel_containment": tumor_inside / tumor_total if tumor_total else np.nan,
            "positive_slice_containment": positive_inside / positive_total if positive_total else np.nan,
            "tumor_pixels": tumor_total, "positive_slices": positive_total}


if REUSE_HISTORY and ORIG_ROI_MANIFEST.is_file():
    shutil.copy2(ORIG_ROI_MANIFEST, TRAIN_ROI_PATH)
    training_rois = pd.read_csv(TRAIN_ROI_PATH)
    print(f"REUSE: training ROI manifest copied ({len(training_rois)} patients).")
else:
    roi_rows = []
    for number, (volume_id, group) in enumerate(
            train_manifest.groupby("volume_id", sort=True), start=1):
        probabilities = predict_volume_liver(group)
        liver_mask = largest_component(probabilities >= 0.50)
        box = padded_bbox(liver_mask, 16)
        scores = score_box(group, box)
        roi_rows.append({"volume_id": int(volume_id), "roi_empty": box is None,
                         "y0": box[0] if box else np.nan, "y1": box[1] if box else np.nan,
                         "x0": box[2] if box else np.nan, "x1": box[3] if box else np.nan,
                         "crop_area_ratio": (((box[1] - box[0]) * (box[3] - box[2])) / (256 * 256)
                                             if box else 0.0), **scores})
        if number % 10 == 0 or number == 104:
            print(f"ROI patients processed: {number}/104")
    training_rois = pd.DataFrame(roi_rows)
    training_rois.to_csv(TRAIN_ROI_PATH, index=False)
    print(f"REBUILD: generated {len(training_rois)} training ROIs.")

display(training_rois.describe(include="all"))

### 3.3 Training ROI gate + audit

In [ ]:
positive_roi_patients = training_rois.loc[training_rois["tumor_pixels"].gt(0)]
training_roi_gate = {
    "minimum_tumor_pixel_containment": float(positive_roi_patients["tumor_pixel_containment"].min()),
    "minimum_positive_slice_containment": float(positive_roi_patients["positive_slice_containment"].min()),
    "empty_training_rois": int(training_rois["roi_empty"].sum()),
    "median_crop_area_ratio": float(training_rois["crop_area_ratio"].median()),
    "maximum_crop_area_ratio": float(training_rois["crop_area_ratio"].max()),
}
training_roi_gate["passed"] = bool(
    training_roi_gate["minimum_tumor_pixel_containment"] >= 0.99
    and training_roi_gate["minimum_positive_slice_containment"] >= 0.99
    and training_roi_gate["empty_training_rois"] == 0)
(OUT["mark_3"] / "training_roi_gate.json").write_text(json.dumps(training_roi_gate, indent=2))

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(training_rois["crop_area_ratio"], bins=20, color="#2878B5")
axes[0].axvline(0.60, linestyle="--", color="#444444")
axes[0].set_title("Training ROI area"); axes[0].set_xlabel("Crop-area ratio")
axes[1].hist(positive_roi_patients["tumor_pixel_containment"],
             bins=np.linspace(0.95, 1.0, 21), color="#4E9F3D")
axes[1].axvline(0.99, linestyle="--", color="#444444")
axes[1].set_title("Tumor-pixel containment")
axes[2].scatter(training_rois["crop_area_ratio"],
                training_rois["tumor_pixel_containment"].fillna(1.0), alpha=0.7)
axes[2].axhline(0.99, linestyle="--", color="#444444")
axes[2].set_title("Containment versus crop burden"); axes[2].set_xlabel("Crop-area ratio")
figure.suptitle("Frozen training ROI audit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_3"] / "training_roi_audit.png", dpi=170, bbox_inches="tight")
plt.show()
display(pd.DataFrame([training_roi_gate]).T.rename(columns={0: "value"}))
assert training_roi_gate["passed"], "STOP: training ROI gate failed."

### 3.4 Select the fixed 16-slice stratified cohort

In [ ]:
ORIG_SELECTED = MARK1_DIR / "mark_3_outputs" / "overfit_selected_slices.csv"
SELECTED_PATH = OUT["mark_3"] / "overfit_selected_slices.csv"
if REUSE_HISTORY and ORIG_SELECTED.is_file():
    shutil.copy2(ORIG_SELECTED, SELECTED_PATH)
    selected_frame = pd.read_csv(SELECTED_PATH)
    print(f"REUSE: cohort copied ({len(selected_frame)} slices).")
else:
    positive = train_manifest.loc[train_manifest["tumor_pixels"].gt(0)].copy()
    positive["size_quartile"] = pd.qcut(positive["tumor_pixels"], 4,
                                        labels=["Q1", "Q2", "Q3", "Q4"])
    rng = np.random.default_rng(SEED)
    selected_rows, used_patients = [], set()
    for quartile in ["Q1", "Q2", "Q3", "Q4"]:
        candidates = positive.loc[positive["size_quartile"].eq(quartile)].copy()
        candidates = candidates.sample(frac=1.0, random_state=SEED)
        chosen = []
        for row in candidates.itertuples(index=False):
            if row.volume_id not in used_patients or len(chosen) >= 3:
                chosen.append(row)
                used_patients.add(row.volume_id)
            if len(chosen) == 4:
                break
        selected_rows.extend(chosen)
    selected_frame = pd.DataFrame([row._asdict() for row in selected_rows])
    selected_frame.to_csv(SELECTED_PATH, index=False)
    print(f"REBUILD: selected {len(selected_frame)} slices.")

assert len(selected_frame) == 16 and selected_frame["volume_id"].nunique() >= 8
display(selected_frame[["sample_id", "volume_id", "slice_index", "tumor_pixels", "size_quartile"]])

### 3.5 Build source-HU ROI tensors and verify round-trip geometry

In [ ]:
import nibabel as nib

roi_index = training_rois.set_index("volume_id")
volume_cache = {}


def source_hu_slice(row):
    volume_id = int(row.volume_id)
    if volume_id not in volume_cache:
        volume_cache[volume_id] = nib.load(str(row.source_volume_path))
    return np.asanyarray(volume_cache[volume_id].dataobj[:, :, int(row.slice_index)]).astype(np.float32)


WINDOWS_3 = {"broad": (-160.0, 240.0), "liver": (-20.0, 140.0), "narrow": (20.0, 120.0)}
CHANNEL_CONFIGS = {"broad_1ch": ["broad"], "broad_liver_2ch": ["broad", "liver"],
                   "broad_liver_narrow_3ch": ["broad", "liver", "narrow"]}

tensors = {name: [] for name in CHANNEL_CONFIGS}
targets = []
roundtrip_rows, preview_rows = [], []
for row in selected_frame.itertuples(index=False):
    box_row = roi_index.loc[int(row.volume_id)]
    y0, y1, x0, x1 = [int(box_row[key]) for key in ("y0", "y1", "x0", "x1")]
    hu_native = source_hu_slice(row)
    channels_256 = {name: resize_float(window_hu(hu_native, (lower, upper)))
                    for name, (lower, upper) in WINDOWS_3.items()}
    with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
        tumor_full = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
    target_roi = resize_mask(tumor_full[y0:y1, x0:x1])
    targets.append(target_roi[None].astype(np.float32))
    for config_name, channel_names in CHANNEL_CONFIGS.items():
        crop_channels = [resize_float(channels_256[name][y0:y1, x0:x1])
                         for name in channel_names]
        tensors[config_name].append(np.stack(crop_channels))
    restored_crop = resize_mask(target_roi, size=(x1 - x0, y1 - y0))
    restored = np.zeros((256, 256), dtype=bool)
    restored[y0:y1, x0:x1] = restored_crop
    intersection = int((restored & tumor_full).sum())
    roundtrip_rows.append({"sample_id": row.sample_id, "volume_id": int(row.volume_id),
                           "roundtrip_dice": float((2 * intersection + 1e-6)
                                                   / (restored.sum() + tumor_full.sum() + 1e-6)),
                           "box": [y0, y1, x0, x1]})
    preview_rows.append((row, channels_256, tumor_full, (y0, y1, x0, x1)))

targets = torch.from_numpy(np.stack(targets)).float()
tensors = {name: torch.from_numpy(np.stack(values)).float() for name, values in tensors.items()}
roundtrip_metrics = pd.DataFrame(roundtrip_rows)
roundtrip_metrics.to_csv(OUT["mark_3"] / "roundtrip_geometry_metrics.csv", index=False)
assert roundtrip_metrics["roundtrip_dice"].min() >= 0.98
for name, values in tensors.items():
    assert values.shape[0] == 16 and values.shape[2:] == (256, 256) and torch.isfinite(values).all()
print("PASS: ROI tensors and full-image round-trip geometry verified.")
print(roundtrip_metrics.describe())

figure, axes = plt.subplots(4, 5, figsize=(18, 14))
for row_axes, (row, channels, tumor, box) in zip(axes, preview_rows[:4]):
    y0, y1, x0, x1 = box
    panels = [(channels["broad"], "Broad full"), (channels["liver"], "Liver full"),
              (channels["narrow"], "Narrow full"),
              (resize_float(channels["broad"][y0:y1, x0:x1]), "Broad ROI"),
              (resize_mask(tumor[y0:y1, x0:x1]), "Tumor ROI")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(f"{row.sample_id}\n{row.size_quartile}", fontsize=9)
figure.suptitle("Source-HU ROI tensor audit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_3"] / "roi_tensor_audit.png", dpi=170, bbox_inches="tight")
plt.show()

### 3.6 Warm start, stable loss, and overfit metrics

In [ ]:
from src.framework.losses.focal_dice import FocalDiceLoss

source_state = checkpoint["model_state"]
loss_function = FocalDiceLoss(focal_alpha=0.75, focal_gamma=2.0,
                              focal_weight=0.5, dice_weight=0.5)


def build_tumor_model(in_channels):
    model = MobileNetV2UNet(in_channels=in_channels, out_channels=1, pretrained=False)
    target_state = model.state_dict()
    for key, value in source_state.items():
        if key in target_state and target_state[key].shape == value.shape:
            target_state[key] = value.clone()
    source_first = source_state["enc_0.0.weight"]
    target_state["enc_0.0.weight"] = source_first.repeat(1, in_channels, 1, 1) / in_channels
    target_state["final.weight"] = source_state["final.weight"][1:2].clone()
    target_state["final.bias"] = source_state["final.bias"][1:2].clone()
    model.load_state_dict(target_state, strict=True)
    return model.to(DEVICE)


def freeze_batchnorm_running_stats(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def hard_metrics(logits, truth):
    prediction = torch.sigmoid(logits) >= 0.50
    target = truth >= 0.5
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    true = int(target.sum())
    dice = (2 * intersection + 1e-6) / (predicted + true + 1e-6)
    pred_per_slice = prediction.sum(dim=(1, 2, 3))
    true_per_slice = target.sum(dim=(1, 2, 3))
    positive_empty = int(((true_per_slice > 0) & (pred_per_slice == 0)).sum())
    return {"hard_micro_dice": float(dice),
            "positive_predicted_empty_pct": 100 * positive_empty / len(truth)}

### 3.7 Overfit ablation (reuse frozen history or retrain)

In [ ]:
ORIG_HIST = MARK1_DIR / "mark_3_outputs" / "overfit_history.csv"
ORIG_COMP = MARK1_DIR / "mark_3_outputs" / "overfit_channel_comparison.csv"
HIST_PATH = OUT["mark_3"] / "overfit_history.csv"
COMP_PATH = OUT["mark_3"] / "overfit_channel_comparison.csv"

if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    shutil.copy2(ORIG_COMP, COMP_PATH)
    history_frame = pd.read_csv(HIST_PATH)
    comparison = pd.read_csv(COMP_PATH)
    trained_models = {}
    print(f"REUSE: overfit history copied ({len(history_frame)} rows).")
else:
    histories, final_metrics, trained_models = [], [], {}
    for config_name, input_tensor in tensors.items():
        print(f"\n=== {config_name} ===")
        model = build_tumor_model(input_tensor.shape[1])
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
        loader_generator = torch.Generator().manual_seed(SEED)
        loader = DataLoader(TensorDataset(input_tensor, targets), batch_size=4,
                            shuffle=True, generator=loader_generator, num_workers=0)
        best_dice = 0.0
        for epoch in range(1, 161):
            model.train()
            freeze_batchnorm_running_stats(model)
            epoch_loss, gradient_finite = 0.0, True
            for images, truth in loader:
                images, truth = images.to(DEVICE), truth.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits = model(images)
                loss = loss_function(logits, truth)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss for {config_name} epoch {epoch}")
                loss.backward()
                gradient_finite &= all(p.grad is None or torch.isfinite(p.grad).all()
                                       for p in model.parameters())
                if not gradient_finite:
                    raise FloatingPointError("Non-finite gradient.")
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
                epoch_loss += float(loss) * len(images)
            model.eval()
            with torch.inference_mode():
                metrics = hard_metrics(model(input_tensor.to(DEVICE)), targets.to(DEVICE))
            histories.append({"configuration": config_name,
                              "channels": int(input_tensor.shape[1]), "epoch": epoch,
                              "loss": epoch_loss / len(input_tensor),
                              "gradient_finite": gradient_finite, **metrics})
            best_dice = max(best_dice, metrics["hard_micro_dice"])
            if epoch == 1 or epoch % 10 == 0:
                print(f"epoch={epoch:03d} loss={histories[-1]['loss']:.4f} "
                      f"dice={metrics['hard_micro_dice']:.4f} "
                      f"empty={metrics['positive_predicted_empty_pct']:.1f}%")
            if metrics["hard_micro_dice"] >= 0.90 and metrics["positive_predicted_empty_pct"] == 0:
                break
        trained_models[config_name] = model.cpu()
        final_metrics.append({"configuration": config_name,
                              "channels": int(input_tensor.shape[1]),
                              "epochs_completed": epoch, "best_hard_micro_dice": best_dice,
                              "final_hard_micro_dice": metrics["hard_micro_dice"],
                              "positive_predicted_empty_pct": metrics["positive_predicted_empty_pct"],
                              "passed": bool(metrics["hard_micro_dice"] >= 0.90
                                             and metrics["positive_predicted_empty_pct"] == 0)})
        torch.save({"model_state": model.state_dict(), "configuration": config_name,
                    "channels": CHANNEL_CONFIGS[config_name],
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256},
                   OUT["mark_3"] / f"{config_name}_overfit.pth")
    history_frame = pd.DataFrame(histories)
    comparison = pd.DataFrame(final_metrics)
    history_frame.to_csv(HIST_PATH, index=False)
    comparison.to_csv(COMP_PATH, index=False)
    print("REBUILD: overfit ablation completed.")

display(comparison)

### 3.8 Convergence dashboard, selection, and prediction review

In [ ]:
passing = comparison.loc[comparison["passed"]].sort_values(
    ["channels", "epochs_completed", "final_hard_micro_dice"],
    ascending=[True, True, False])
selected_configuration = (passing.iloc[0] if not passing.empty else
                          comparison.sort_values("best_hard_micro_dice", ascending=False).iloc[0])

figure, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for name, group in history_frame.groupby("configuration"):
    axes[0].plot(group["epoch"], group["loss"], label=name)
    axes[1].plot(group["epoch"], group["hard_micro_dice"], label=name)
    axes[2].plot(group["epoch"], group["positive_predicted_empty_pct"], label=name)
axes[0].set_title("Overfit loss")
axes[1].axhline(0.90, linestyle="--", color="#444444"); axes[1].set_title("Hard micro-Dice")
axes[2].axhline(0, linestyle="--", color="#444444")
axes[2].set_title("Positive predicted-empty (%)")
for axis in axes:
    axis.set_xlabel("Epoch"); axis.legend(fontsize=8)
figure.suptitle("Controlled channel-ablation overfit", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_3"] / "overfit_convergence_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()
print("Selected:", selected_configuration.to_dict())

# Prediction review (loads saved checkpoint for the selected config)
selected_name = selected_configuration["configuration"]
if selected_name in trained_models:
    selected_model = trained_models[selected_name].to(DEVICE).eval()
else:
    saved = torch.load(OUT["mark_3"] / f"{selected_name}_overfit.pth",
                       map_location="cpu", weights_only=False)
    selected_model = MobileNetV2UNet(in_channels=CHANNEL_CONFIGS[selected_name].__len__(),
                                     out_channels=1, pretrained=False)
    selected_model.load_state_dict(saved["model_state"], strict=True)
    selected_model.to(DEVICE).eval()
selected_inputs = tensors[selected_name].to(DEVICE)
with torch.inference_mode():
    selected_probabilities = torch.sigmoid(selected_model(selected_inputs)).cpu()
predictions = selected_probabilities >= 0.50

figure, axes = plt.subplots(4, 4, figsize=(14, 14))
for row_axes, index in zip(axes, [0, 4, 8, 12]):
    panels = [(selected_inputs[index, 0].cpu(), "Broad ROI"),
              (targets[index, 0], "Expected tumor"),
              (selected_probabilities[index, 0], "Tumor probability"),
              (predictions[index, 0], "Generated tumor")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="magma" if "probability" in title else "gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(selected_frame.iloc[index]["sample_id"], fontsize=8)
figure.suptitle(f"Overfit predictions — {selected_name}", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_3"] / "overfit_prediction_review.png", dpi=170, bbox_inches="tight")
plt.show()

### 3.9 Write the Mark 3 gate

In [ ]:
overfit_passed = bool(selected_configuration["passed"])
geometry_passed = bool(roundtrip_metrics["roundtrip_dice"].min() >= 0.98)
full_gate_passed = bool(training_roi_gate["passed"] and overfit_passed and geometry_passed)

m3_gate = {
    "status": "mark_3_overfit_pass" if full_gate_passed else "mark_3_overfit_fail",
    "training_roi_gate_passed": bool(training_roi_gate["passed"]),
    "overfit_gate_passed": overfit_passed,
    "geometry_gate_passed": geometry_passed,
    "selected_configuration": selected_configuration.to_dict(),
    "selected_channels": CHANNEL_CONFIGS[selected_configuration["configuration"]],
    "minimum_training_tumor_containment": training_roi_gate["minimum_tumor_pixel_containment"],
    "minimum_roundtrip_dice": float(roundtrip_metrics["roundtrip_dice"].min()),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "test_images_accessed": False,
    "decision": ("PROCEED_TO_3_TO_5_EPOCH_TWO_STAGE_VALIDATION_SMOKE" if full_gate_passed
                 else "STOP_AND_REPAIR_ROI_GEOMETRY_OR_INPUT_REPRESENTATION"),
    "next_notebook": "mark_4_two_stage_validation_smoke" if full_gate_passed else "mark_3_revision",
}
(OUT["mark_3"] / "mark_3_gate_result.json").write_text(json.dumps(m3_gate, indent=2))
display(pd.DataFrame([m3_gate]).T.rename(columns={0: "value"}))
print(m3_gate["decision"])

# ---- Reproduction check against the original gate ----
orig_m3 = json.loads((MARK1_DIR / "mark_3_outputs" / "mark_3_gate_result.json").read_text())
orig_sel = orig_m3["selected_configuration"]
recomputed_sel = m3_gate["selected_configuration"]
diffs = {k: abs(float(recomputed_sel[k]) - float(orig_sel[k])) for k in
         ["best_hard_micro_dice", "final_hard_micro_dice", "positive_predicted_empty_pct"]}
print("Mark 3 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 3 overfit gate drifted from the original!"
assert m3_gate["status"] == orig_m3["status"]
print("PASS: Mark 3 gate matches the original mark_3_gate_result.json.")

# Part 4 — Mark 4: Two-Stage ROI Validation Smoke Test

**Original:** `mark 1/mark_4_two_stage_validation_smoke.ipynb`

## Question

After the 16-slice overfit proof, does the same frozen pipeline generalize across **all** training and
validation patients in a bounded 5-epoch smoke run?

## Key finding (reproduced)

- **Smoke gate: FAILED for continuation** (5/6 temporary targets at best epoch 5).
- Best epoch 5: mean patient Dice 0.3639, V104 0.0672, V116 0.0108, Q1 42.59%,
  **positive predicted-empty 36.85%** (> 35% target), empty-slice FP 3.42%.
- The recall failure is concentrated on tumor-positive slices → Mark 4B probes threshold calibration.

## Contract

- Frozen pipeline: ROI (0.50, `largest_3d`, pad 16) + broad window `[-160,240]`, tumor threshold 0.50.
- Fresh warm start from the epoch-8 multi-task checkpoint (the 16-slice memorization checkpoint is NOT used).
- Temporary continuation targets: mean patient Dice ≥ 0.3329, V104 ≥ 0.05, V116 ≥ 0.01,
  Q1 ≥ 35%, positive empty ≤ 35%, empty FP ≤ 20%.
- Test split locked.

### 4.1 Freeze training + validation ROI manifests

In [ ]:
import shutil

training_rois = pd.read_csv(OUT["mark_3"] / "training_roi_manifest.csv")
assert len(training_rois) == 104 and not training_rois["roi_empty"].astype(bool).any()

mark2_roi_rows = pd.read_csv(OUT["mark_2"] / "roi_patient_results.csv")
validation_rois = mark2_roi_rows.loc[
    mark2_roi_rows["liver_threshold"].eq(0.50)
    & mark2_roi_rows["padding"].eq(16)
    & mark2_roi_rows["component_mode"].eq("largest_3d")].copy()
assert len(validation_rois) == 13 and not validation_rois["roi_empty"].astype(bool).any()
validation_rois.to_csv(OUT["mark_4"] / "validation_roi_manifest.csv", index=False)

roi_summary = pd.DataFrame([
    {"split": "train", "patients": len(training_rois),
     "median_area": training_rois["crop_area_ratio"].median(),
     "max_area": training_rois["crop_area_ratio"].max(),
     "empty_rois": int(training_rois["roi_empty"].sum())},
    {"split": "validation", "patients": len(validation_rois),
     "median_area": validation_rois["crop_area_ratio"].median(),
     "max_area": validation_rois["crop_area_ratio"].max(),
     "empty_rois": int(validation_rois["roi_empty"].sum())},
])
display(roi_summary)

### 4.2 Build the ROI dataset and audit PNG/source-HU parity

In [ ]:
import cv2


class SynchronizedROIAugment:
    def __call__(self, image, mask):
        if random.random() < 0.30:
            image, mask = np.fliplr(image).copy(), np.fliplr(mask).copy()
        if random.random() < 0.50:
            height, width = image.shape
            matrix = cv2.getRotationMatrix2D(
                (width / 2, height / 2), random.uniform(-8, 8),
                random.uniform(0.97, 1.03))
            image = cv2.warpAffine(image, matrix, (width, height), flags=cv2.INTER_LINEAR)
            mask = cv2.warpAffine(mask.astype(np.uint8), matrix, (width, height),
                                  flags=cv2.INTER_NEAREST) > 0
        if random.random() < 0.35:
            image = np.clip(image + np.random.normal(0, 0.015, image.shape), 0, 1)
        return image.astype(np.float32), mask.astype(np.float32)


class ROISliceDataset(Dataset):
    def __init__(self, rows, roi_frame, augment=None):
        self.rows = rows.reset_index(drop=True)
        self.rois = roi_frame.set_index("volume_id")
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        box = self.rois.loc[int(row.volume_id)]
        y0, y1, x0, x1 = [int(box[key]) for key in ("y0", "y1", "x0", "x1")]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            truth_full = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        image_roi = resize_float(image[y0:y1, x0:x1])
        truth_roi = resize_mask(truth_full[y0:y1, x0:x1])
        if self.augment is not None:
            image_roi, truth_roi = self.augment(image_roi, truth_roi)
        return {"image": torch.from_numpy(image_roi[None]).float(),
                "mask": torch.from_numpy(np.asarray(truth_roi)[None]).float(),
                "sample_id": str(row.sample_id), "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index),
                "true_pixels_full": int(row.tumor_pixels),
                "box": torch.tensor([y0, y1, x0, x1], dtype=torch.int32)}


train_dataset = ROISliceDataset(train_manifest, training_rois,
                                augment=SynchronizedROIAugment())
validation_dataset = ROISliceDataset(validation_manifest, validation_rois, augment=None)
print(f"Train={len(train_dataset):,} | Validation={len(validation_dataset):,}")

import nibabel as nib
audit_rows = train_manifest.sample(32, random_state=SEED)
parity_rows, loaded_volumes = [], {}
for row in audit_rows.itertuples(index=False):
    if row.source_volume_path not in loaded_volumes:
        loaded_volumes[row.source_volume_path] = nib.load(str(row.source_volume_path))
    hu = np.asanyarray(
        loaded_volumes[row.source_volume_path].dataobj[:, :, int(row.slice_index)]
    ).astype(np.float32)
    regenerated = resize_float(np.clip((hu - BROAD_WINDOW[0]) / (BROAD_WINDOW[1] - BROAD_WINDOW[0]), 0, 1))
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        stored = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    parity_rows.append({"sample_id": row.sample_id,
                        "mean_absolute_error": float(np.mean(np.abs(regenerated - stored))),
                        "max_absolute_error": float(np.max(np.abs(regenerated - stored)))})
parity = pd.DataFrame(parity_rows)
parity.to_csv(OUT["mark_4"] / "broad_png_source_parity.csv", index=False)
print(parity.describe())
assert parity["mean_absolute_error"].median() <= 0.01

preview = [train_dataset[i] for i in [0, 1000, 10000, 30000]]
figure, axes = plt.subplots(4, 2, figsize=(9, 16))
for row_axes, item in zip(axes, preview):
    row_axes[0].imshow(item["image"][0], cmap="gray", vmin=0, vmax=1)
    row_axes[0].set_title(f"{item['sample_id']} broad ROI")
    row_axes[1].imshow(item["mask"][0], cmap="gray", vmin=0, vmax=1)
    row_axes[1].set_title("Tumor target")
    for axis in row_axes:
        axis.axis("off")
figure.suptitle("Full-dataset ROI geometry audit", fontsize=16, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_4"] / "roi_dataset_audit.png", dpi=170, bbox_inches="tight")
plt.show()

### 4.3 Patient-aware stratified loaders + fresh warm start

In [ ]:
volume_counts = train_manifest["volume_id"].value_counts()
weights = train_manifest["volume_id"].map(
    lambda v: 1.0 / volume_counts.loc[v]).to_numpy(dtype=np.float64)
weights *= np.where(train_manifest["tumor_pixels"].to_numpy() > 0, 3.0, 1.0)
weights /= weights.mean()
sampler_generator = torch.Generator().manual_seed(SEED)
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
                                num_samples=len(train_dataset), replacement=True,
                                generator=sampler_generator)
train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler,
                          num_workers=0, pin_memory=torch.cuda.is_available())
validation_loader = DataLoader(validation_dataset, batch_size=8, shuffle=False,
                               num_workers=0, pin_memory=torch.cuda.is_available())

source_payload = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)
source_state = source_payload["model_state"]
model = MobileNetV2UNet(in_channels=1, out_channels=1, pretrained=False)
target_state = model.state_dict()
for key, value in source_state.items():
    if key in target_state and target_state[key].shape == value.shape:
        target_state[key] = value.clone()
target_state["final.weight"] = source_state["final.weight"][1:2].clone()
target_state["final.bias"] = source_state["final.bias"][1:2].clone()
model.load_state_dict(target_state, strict=True)
model.to(DEVICE)

loss_function = FocalDiceLoss(focal_alpha=0.75, focal_gamma=2.0,
                              focal_weight=0.5, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=3e-5)
print("PASS: fresh warm start from epoch-8 multi-task checkpoint.")

### 4.4 Full-image validation metrics (ROI → 256×256 inverse mapping)

In [ ]:
def evaluate(model):
    model.eval()
    patient_accumulators, slice_rows, validation_loss = {}, [], 0.0
    with torch.inference_mode():
        for batch in validation_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            logits = model(images)
            validation_loss += float(loss_function(logits, masks)) * len(images)
            probabilities = torch.sigmoid(logits).cpu().numpy()[:, 0]
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                slice_index = int(batch["slice_index"][index])
                box = batch["box"][index].numpy()
                probability_full = probability_to_full(probabilities[index], box)
                prediction = probability_full >= 0.50
                manifest_row = validation_manifest.loc[
                    validation_manifest["sample_id"].eq(sample_id)].iloc[0]
                with Image.open(DATASET_ROOT / manifest_row["tumor_mask_path"]) as handle:
                    truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
                intersection = int((prediction & truth).sum())
                predicted, true = int(prediction.sum()), int(truth.sum())
                values = patient_accumulators.setdefault(volume_id, {
                    "intersection": 0, "predicted": 0, "true": 0,
                    "positive": 0, "detected": 0, "positive_empty": 0,
                    "empty": 0, "empty_fp": 0})
                values["intersection"] += intersection
                values["predicted"] += predicted
                values["true"] += true
                if true > 0:
                    values["positive"] += 1
                    values["detected"] += int(intersection > 0)
                    values["positive_empty"] += int(predicted == 0)
                else:
                    values["empty"] += 1
                    values["empty_fp"] += int(predicted > 0)
                slice_rows.append({"sample_id": sample_id, "volume_id": volume_id,
                                   "slice_index": slice_index, "true_pixels": true,
                                   "predicted_pixels": predicted,
                                   "intersection_pixels": intersection,
                                   "dice": (2 * intersection + 1e-6) / (predicted + true + 1e-6)})
    patient_rows = [{"volume_id": v, "true_pixels": x["true"],
                     "predicted_pixels": x["predicted"],
                     "micro_dice": (2 * x["intersection"] + 1e-6) / (x["predicted"] + x["true"] + 1e-6),
                     "positive_predicted_empty_pct": 100 * x["positive_empty"] / max(x["positive"], 1),
                     "empty_slice_false_positive_pct": 100 * x["empty_fp"] / max(x["empty"], 1)}
                    for v, x in sorted(patient_accumulators.items())]
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = pd.qcut(positive_slices["true_pixels"], 4,
                                               labels=["Q1", "Q2", "Q3", "Q4"])
    size_rows = [{"size_quartile": str(q), "slices": len(g),
                  "mean_dice": g["dice"].mean(),
                  "detected_pct": 100 * (g["intersection_pixels"] > 0).mean(),
                  "predicted_empty_pct": 100 * (g["predicted_pixels"] == 0).mean()}
                 for q, g in positive_slices.groupby("size_quartile", observed=True)]
    sizes = pd.DataFrame(size_rows)
    totals = patient_accumulators.values()
    result = {
        "validation_loss": validation_loss / len(validation_dataset),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": float(sizes.set_index("size_quartile").loc["Q1", "detected_pct"]),
        "positive_predicted_empty_pct": float(100 * sum(v["positive_empty"] for v in totals)
                                              / max(sum(v["positive"] for v in totals), 1)),
        "empty_slice_false_positive_pct": float(100 * sum(v["empty_fp"] for v in totals)
                                                / max(sum(v["empty"] for v in totals), 1)),
    }
    return result, patients, slices, sizes

### 4.5 Smoke training (reuse frozen history or retrain 5 epochs)

In [ ]:
import subprocess

ORIG_HIST = MARK1_DIR / "mark_4_outputs" / "mark_4_history.csv"
HIST_PATH = OUT["mark_4"] / "mark_4_history.csv"


def gpu_temperature():
    if not torch.cuda.is_available():
        return np.nan
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True)
        return float(result.stdout.splitlines()[0])
    except Exception:
        return np.nan


def freeze_batchnorm_running_stats(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    history = json.loads(pd.read_csv(HIST_PATH).to_json(orient="records"))
    print(f"REUSE: Mark 4 history copied ({len(history)} epochs).")
else:
    history, best_score, best_epoch, best_result = [], -np.inf, None, None
    for epoch in range(1, 6):
        model.train()
        freeze_batchnorm_running_stats(model)
        train_loss, gradients_finite = 0.0, True
        for batch in train_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = loss_function(logits, masks)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite training loss.")
            loss.backward()
            gradients_finite &= all(p.grad is None or torch.isfinite(p.grad).all()
                                    for p in model.parameters())
            if not gradients_finite:
                raise FloatingPointError("Non-finite gradient.")
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += float(loss) * len(images)
        result, patient_metrics, slice_metrics, size_metrics = evaluate(model)
        scheduler.step()
        record = {"epoch": epoch, "train_loss": train_loss / len(train_dataset), **result,
                  "learning_rate": optimizer.param_groups[0]["lr"],
                  "end_temperature_c": gpu_temperature(), "gradients_finite": gradients_finite}
        history.append(record)
        pd.DataFrame(history).to_csv(HIST_PATH, index=False)
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(), "history": history,
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
                    "roi_rule": {"liver_threshold": 0.50, "padding": 16,
                                 "component_mode": "largest_3d"},
                    "input_window_hu": list(BROAD_WINDOW)},
                   OUT["mark_4"] / "mark_4_last.pth")
        if result["mean_patient_dice"] > best_score:
            best_score = result["mean_patient_dice"]
            best_epoch = epoch
            best_result = result
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(), "history": history,
                        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                        "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256},
                       OUT["mark_4"] / "mark_4_best.pth")
            patient_metrics.to_csv(OUT["mark_4"] / "best_validation_patient_metrics.csv", index=False)
            slice_metrics.to_csv(OUT["mark_4"] / "best_validation_per_slice.csv", index=False)
            size_metrics.to_csv(OUT["mark_4"] / "best_validation_size_metrics.csv", index=False)
        print(f"epoch={epoch} train={record['train_loss']:.4f} "
              f"patient={record['mean_patient_dice']:.4f} "
              f"V104={record['volume_104_dice']:.4f} V116={record['volume_116_dice']:.4f} "
              f"Q1={record['q1_detected_pct']:.1f}% emptyFP={record['empty_slice_false_positive_pct']:.1f}%")
    print("REBUILD: 5-epoch smoke training completed.")

history_frame = pd.DataFrame(history)
display(history_frame)

### 4.6 Smoke-test trajectories + best full-image predictions

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
axes[0, 0].plot(history_frame["epoch"], history_frame["train_loss"], marker="o", label="Train")
axes[0, 0].set_title("Training loss"); axes[0, 0].legend()
axes[0, 1].plot(history_frame["epoch"], history_frame["mean_patient_dice"], marker="o")
axes[0, 1].axhline(CONTINUATION_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 1].set_title("Mean patient Dice")
axes[0, 2].plot(history_frame["epoch"], history_frame["volume_104_dice"], marker="o", label="V104")
axes[0, 2].plot(history_frame["epoch"], history_frame["volume_116_dice"], marker="s", label="V116")
axes[0, 2].legend(); axes[0, 2].set_title("Focus-patient Dice")
axes[1, 0].plot(history_frame["epoch"], history_frame["q1_detected_pct"], marker="o")
axes[1, 0].axhline(CONTINUATION_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[1, 0].set_title("Q1 detection (%)")
axes[1, 1].plot(history_frame["epoch"], history_frame["positive_predicted_empty_pct"],
                marker="o", label="Positive empty")
axes[1, 1].plot(history_frame["epoch"], history_frame["empty_slice_false_positive_pct"],
                marker="s", label="Empty FP")
axes[1, 1].legend(); axes[1, 1].set_title("Slice error rates (%)")
if "end_temperature_c" in history_frame:
    axes[1, 2].plot(history_frame["epoch"], history_frame["end_temperature_c"],
                    marker="o", color="#F28E2B")
    axes[1, 2].axhline(84, linestyle="--", color="#444")
    axes[1, 2].set_title("GPU temperature (C)")
for axis in axes.flat:
    axis.set_xlabel("Epoch")
figure.suptitle("Mark 4 two-stage validation smoke", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_4"] / "mark_4_smoke_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

# Best-epoch full-image predictions (uses the frozen Mark 4 best checkpoint)
best_row = history_frame.loc[history_frame["mean_patient_dice"].idxmax()]
best_epoch = int(best_row["epoch"])
if REUSE_HISTORY:
    best_payload = torch.load(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
                              map_location="cpu", weights_only=False)
    model.load_state_dict(best_payload["model_state"], strict=True)
model.eval()
focus_rows = []
for volume_id in [104, 116, 108, 109]:
    candidates = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)
        & validation_manifest["tumor_pixels"].gt(0)]
    focus_rows.append(candidates.nlargest(1, "tumor_pixels").iloc[0])

figure, axes = plt.subplots(4, 4, figsize=(15, 15))
for row_axes, row in zip(axes, focus_rows):
    dataset_index = int(validation_manifest.index[
        validation_manifest["sample_id"].eq(row.sample_id)][0])
    item = validation_dataset[dataset_index]
    with torch.inference_mode():
        probability_roi = torch.sigmoid(
            model(item["image"][None].to(DEVICE)))[0, 0].cpu().numpy()
    probability_full = probability_to_full(probability_roi, item["box"].numpy())
    prediction = probability_full >= 0.50
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
        truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
    panels = [(image, "Full CT"), (truth, "Expected tumor"),
              (probability_full, "Full probability"), (prediction, "Generated tumor")]
    for axis, (panel, title) in zip(row_axes, panels):
        axis.imshow(panel, cmap="magma" if "probability" in title else "gray", vmin=0, vmax=1)
        axis.set_title(title); axis.axis("off")
    row_axes[0].set_ylabel(f"V{int(row.volume_id)}", fontsize=10)
figure.suptitle(f"Best epoch {best_epoch} full-image predictions", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_4"] / "best_full_image_predictions.png", dpi=170, bbox_inches="tight")
plt.show()

### 4.7 Apply the continuation gate + write the Mark 4 gate

In [ ]:
best_record = history_frame.loc[history_frame["mean_patient_dice"].idxmax()].to_dict()
passes = target_passes(best_record, CONTINUATION_TARGETS)
passes["gradients_finite"] = bool(best_record.get("gradients_finite", True))
continuation_passed = all(passes.values())
final_passes = target_passes(best_record, FINAL_TARGETS)

m4_gate = {
    "status": "mark_4_smoke_pass" if continuation_passed else "mark_4_smoke_fail",
    "best_epoch": int(best_record["epoch"]),
    "best_metrics": {key: float(best_record[key]) for key in CONTINUATION_TARGETS},
    "continuation_targets": CONTINUATION_TARGETS,
    "continuation_passes": passes,
    "continuation_gate_passed": continuation_passed,
    "final_validation_targets": FINAL_TARGETS,
    "final_targets_passed": final_passes,
    "all_final_targets_passed": all(final_passes.values()),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_checkpoint_sha256": EXPECTED_SOURCE_CHECKPOINT_SHA256,
    "overfit_checkpoint_used_for_initialization": False,
    "test_images_accessed": False,
    "decision": ("PROCEED_TO_BOUNDED_LONGER_TWO_STAGE_VALIDATION_RUN" if continuation_passed
                 else "STOP_AND_DIAGNOSE_TWO_STAGE_SMOKE_FAILURE"),
    "next_notebook": ("mark_5_two_stage_bounded_continuation" if continuation_passed
                      else "mark_4_failure_diagnostics"),
}
(OUT["mark_4"] / "mark_4_gate_result.json").write_text(json.dumps(m4_gate, indent=2))
expected_actual = pd.DataFrame([
    {"metric": key, "actual": best_record[key],
     "continuation_target": CONTINUATION_TARGETS[key], "continuation_passed": passes[key],
     "final_target": FINAL_TARGETS[key], "final_passed": final_passes[key]}
    for key in CONTINUATION_TARGETS])
expected_actual.to_csv(OUT["mark_4"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([m4_gate]).T.rename(columns={0: "value"}))
display(expected_actual)
print(m4_gate["decision"])

# ---- Reproduction check against the original gate ----
orig_m4 = json.loads((MARK1_DIR / "mark_4_outputs" / "mark_4_gate_result.json").read_text())
diffs = {k: abs(float(m4_gate["best_metrics"][k]) - float(orig_m4["best_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4 reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4 gate drifted from the original!"
assert m4_gate["status"] == orig_m4["status"] and m4_gate["best_epoch"] == orig_m4["best_epoch"]
print("PASS: Mark 4 gate matches the original mark_4_gate_result.json.")

# Part 5 — Mark 4B: ROI Probability Calibration and Failure Diagnostics

**Original:** `mark 1/mark_4b_roi_probability_diagnostics.ipynb`

## Question

Mark 4 missed the positive predicted-empty gate by ~2 percentage points. Does the epoch-5 model
already contain **usable sub-0.50 tumour probabilities** (fixable by thresholding) or is a training
change required?

## Key finding (reproduced)

- **No threshold passed the temporary continuation gate** — positive predicted-empty stays between
  36.3% and 37.0% across the whole grid.
- Selected global threshold 0.60: mean patient Dice 0.3646, V104 0.0646, V116 0.0104,
  Q1 42.59%, positive empty 37.04% (fail), empty FP 3.40% → **5/6 targets**.
- Threshold-only tuning cannot fix the recall failure → Mark 4C changes one causal factor at a time.

## Contract

- One global threshold for every validation patient (no per-patient/per-size thresholds).
- Threshold grid: 0.05, 0.10, 0.15, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70.
- Temporary targets identical to Mark 4 continuation targets. Test split locked.

### 5.1 Verify the Mark 4 evidence and load the frozen checkpoint

In [ ]:
import shutil

mark4_gate = json.loads((OUT["mark_4"] / "mark_4_gate_result.json").read_text())
assert mark4_gate["status"] == "mark_4_smoke_fail"
assert mark4_gate["best_epoch"] == 5
assert mark4_gate["test_images_accessed"] is False

checkpoint = torch.load(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
                        map_location="cpu", weights_only=False)
assert int(checkpoint["epoch"]) == 5
assert checkpoint["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
assert checkpoint["source_checkpoint_sha256"] == EXPECTED_SOURCE_CHECKPOINT_SHA256
model = MobileNetV2UNet(in_channels=1, out_channels=1, pretrained=False)
model.load_state_dict(checkpoint["model_state"], strict=True)
model.to(DEVICE).eval()
with torch.inference_mode():
    probe = torch.sigmoid(model(torch.zeros(1, 1, 256, 256, device=DEVICE)))
assert probe.shape == (1, 1, 256, 256) and torch.isfinite(probe).all()
print("PASS: frozen Mark 4 best checkpoint loaded.")

### 5.2 Build the frozen validation ROI loader and cache probabilities

In [ ]:
class ValidationROIDataset(Dataset):
    def __init__(self, rows, rois):
        self.rows = rows.reset_index(drop=True)
        self.rois = rois.set_index("volume_id")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        roi = self.rois.loc[int(row.volume_id)]
        y0, y1, x0, x1 = [int(roi[key]) for key in ("y0", "y1", "x0", "x1")]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        image_roi = resize_float(image[y0:y1, x0:x1])
        return {"image": torch.from_numpy(image_roi[None]).float(),
                "sample_id": str(row.sample_id), "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index),
                "box": torch.tensor([y0, y1, x0, x1], dtype=torch.int32)}


validation_rois = pd.read_csv(OUT["mark_4"] / "validation_roi_manifest.csv")
dataset = ValidationROIDataset(validation_manifest, validation_rois)
loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0,
                    pin_memory=torch.cuda.is_available())

ORIG_CACHE = MARK1_DIR / "mark_4b_outputs" / "probability_cache"
CACHE_DIR = OUT["mark_4b"] / "probability_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def save_volume(volume_id, bucket):
    order = np.argsort(bucket["slice_index"])
    np.savez_compressed(CACHE_DIR / f"volume_{volume_id}.npz",
                        **{key: np.asarray(value)[order] for key, value in bucket.items()})


def build_cache():
    current_volume, bucket, processed = None, None, []
    with torch.inference_mode():
        for batch in loader:
            probabilities = torch.sigmoid(
                model(batch["image"].to(DEVICE, non_blocking=True))).cpu().numpy()[:, 0]
            for index, sample_id in enumerate(batch["sample_id"]):
                volume_id = int(batch["volume_id"][index])
                if current_volume is None or volume_id != current_volume:
                    if current_volume is not None:
                        save_volume(current_volume, bucket)
                        processed.append(current_volume)
                    current_volume = volume_id
                    bucket = {"sample_id": [], "slice_index": [],
                              "tumor_probability": [], "tumor_truth": []}
                probability_full = probability_to_full(probabilities[index],
                                                       batch["box"][index].numpy())
                row = validation_manifest.loc[
                    validation_manifest["sample_id"].eq(sample_id)].iloc[0]
                with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
                    truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
                bucket["sample_id"].append(str(sample_id))
                bucket["slice_index"].append(int(batch["slice_index"][index]))
                bucket["tumor_probability"].append(probability_full.astype(np.float16))
                bucket["tumor_truth"].append(truth.astype(np.uint8))
    if current_volume is not None:
        save_volume(current_volume, bucket)
        processed.append(current_volume)
    return processed


existing = sorted(CACHE_DIR.glob("volume_*.npz"))
if REUSE_CACHES and len(existing) == 13:
    print(f"REUSE: {len(existing)} patient caches already present in {CACHE_DIR}.")
elif REUSE_CACHES and len(list(ORIG_CACHE.glob("volume_*.npz"))) == 13:
    for p in ORIG_CACHE.glob("volume_*.npz"):
        shutil.copy2(p, CACHE_DIR / p.name)
    print("REUSE: copied 13 frozen caches from mark_4b_outputs/probability_cache.")
else:
    processed = build_cache()
    print(f"REBUILD: cached {len(processed)} volumes.")

m4b_cache, coverage, stats = {}, [], []
for path in sorted(CACHE_DIR.glob("volume_*.npz")):
    volume_id = int(path.stem.split("_")[-1])
    with np.load(path, allow_pickle=False) as payload:
        item = {key: payload[key] for key in payload.files}
    expected = validation_manifest.loc[
        validation_manifest["volume_id"].eq(volume_id)].sort_values("slice_index")
    assert item["sample_id"].astype(str).tolist() == expected["sample_id"].astype(str).tolist()
    assert np.isfinite(item["tumor_probability"]).all()
    m4b_cache[volume_id] = item
    coverage.append({"volume_id": volume_id, "slices": len(item["sample_id"]),
                     "positive_slices": int(item["tumor_truth"].any(axis=(1, 2)).sum()),
                     "cache_mb": path.stat().st_size / (1024 ** 2)})
    for index, sample_id in enumerate(item["sample_id"]):
        truth = item["tumor_truth"][index].astype(bool)
        probability = item["tumor_probability"][index].astype(np.float32)
        inside = probability[truth]
        stats.append({"sample_id": str(sample_id), "volume_id": volume_id,
                      "slice_index": int(item["slice_index"][index]),
                      "true_pixels": int(truth.sum()),
                      "max_probability": float(probability.max()),
                      "mean_probability_inside_truth": float(inside.mean()) if inside.size else np.nan,
                      "max_probability_inside_truth": float(inside.max()) if inside.size else np.nan})
coverage_frame = pd.DataFrame(coverage)
m4b_stats = pd.DataFrame(stats)
assert coverage_frame["slices"].sum() == 10_685
coverage_frame.to_csv(OUT["mark_4b"] / "cache_coverage.csv", index=False)
m4b_stats.to_csv(OUT["mark_4b"] / "probability_slice_statistics.csv", index=False)
display(coverage_frame)
print("PASS: 13 complete probability caches validated.")

### 5.3 Evaluate the global threshold grid

In [ ]:
THRESHOLD_GRID = np.array([0.05, 0.10, 0.15, 0.20, 0.30, 0.35, 0.40, 0.45,
                           0.50, 0.55, 0.60, 0.70], dtype=np.float32)

positive_size_reference = m4b_stats.loc[m4b_stats["true_pixels"].gt(0),
                                        ["sample_id", "true_pixels"]].copy()
positive_size_reference["size_quartile"] = pd.qcut(
    positive_size_reference["true_pixels"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
size_map = positive_size_reference.set_index("sample_id")["size_quartile"]


def evaluate_threshold(threshold):
    patient_rows, slice_rows = [], []
    total_intersection = total_predicted = total_true = 0
    for volume_id, item in m4b_cache.items():
        truth = item["tumor_truth"].astype(bool)
        prediction = item["tumor_probability"].astype(np.float32) >= threshold
        intersections = (prediction & truth).sum(axis=(1, 2))
        predicted = prediction.sum(axis=(1, 2))
        true = truth.sum(axis=(1, 2))
        total_intersection += int(intersections.sum())
        total_predicted += int(predicted.sum())
        total_true += int(true.sum())
        patient_rows.append({
            "volume_id": volume_id, "true_pixels": int(true.sum()),
            "predicted_pixels": int(predicted.sum()),
            "intersection_pixels": int(intersections.sum()),
            "micro_dice": (2 * intersections.sum() + 1e-6) / (predicted.sum() + true.sum() + 1e-6),
            "positive_predicted_empty_pct": 100 * int(((true > 0) & (predicted == 0)).sum())
                                            / max(int((true > 0).sum()), 1),
            "empty_slice_false_positive_pct": 100 * int(((true == 0) & (predicted > 0)).sum())
                                              / max(int((true == 0).sum()), 1)})
        for index, sample_id in enumerate(item["sample_id"]):
            slice_rows.append({"sample_id": str(sample_id), "volume_id": volume_id,
                               "true_pixels": int(true[index]),
                               "predicted_pixels": int(predicted[index]),
                               "intersection_pixels": int(intersections[index])})
    patients = pd.DataFrame(patient_rows)
    slices = pd.DataFrame(slice_rows)
    positive_patients = patients.loc[patients["true_pixels"].gt(0)]
    positive_slices = slices.loc[slices["true_pixels"].gt(0)].copy()
    positive_slices["size_quartile"] = positive_slices["sample_id"].map(size_map)
    q1 = positive_slices.loc[positive_slices["size_quartile"].eq("Q1")]
    return {
        "threshold": float(threshold),
        "global_dice": (2 * total_intersection + 1e-6) / (total_predicted + total_true + 1e-6),
        "pixel_precision": total_intersection / max(total_predicted, 1),
        "pixel_recall": total_intersection / max(total_true, 1),
        "mean_patient_dice": float(positive_patients["micro_dice"].mean()),
        "median_patient_dice": float(positive_patients["micro_dice"].median()),
        "worst_patient_dice": float(positive_patients["micro_dice"].min()),
        "volume_104_dice": float(patients.set_index("volume_id")["micro_dice"].get(104, np.nan)),
        "volume_116_dice": float(patients.set_index("volume_id")["micro_dice"].get(116, np.nan)),
        "q1_detected_pct": 100 * float((q1["intersection_pixels"] > 0).mean()),
        "positive_predicted_empty_pct": 100 * float(
            (positive_slices["predicted_pixels"] == 0).sum() / len(positive_slices)),
        "empty_slice_false_positive_pct": 100 * float(
            ((slices["true_pixels"] == 0) & (slices["predicted_pixels"] > 0)).sum()
            / max((slices["true_pixels"] == 0).sum(), 1)),
        "predicted_tumor_pixels": total_predicted,
    }, patients, slices


result_rows, patient_frames = [], []
for threshold in THRESHOLD_GRID:
    result, patients, slices = evaluate_threshold(float(threshold))
    result_rows.append(result)
    patient_frames.append(patients.assign(threshold=float(threshold)))
m4b_threshold_results = pd.DataFrame(result_rows)
m4b_patient_metrics = pd.concat(patient_frames, ignore_index=True)
m4b_threshold_results.to_csv(OUT["mark_4b"] / "threshold_results.csv", index=False)
m4b_patient_metrics.to_csv(OUT["mark_4b"] / "threshold_patient_metrics.csv", index=False)
display(m4b_threshold_results)

### 5.4 Apply continuation and final-target gates

In [ ]:
def pass_columns(frame, targets, prefix):
    pass_frame = pd.DataFrame(index=frame.index)
    for key, target in targets.items():
        if key in ("positive_predicted_empty_pct", "empty_slice_false_positive_pct"):
            pass_frame[f"{prefix}_{key}"] = frame[key] <= target
        else:
            pass_frame[f"{prefix}_{key}"] = frame[key] >= target
    return pass_frame


cont_flags = pass_columns(m4b_threshold_results, CONTINUATION_TARGETS, "continue")
final_flags = pass_columns(m4b_threshold_results, FINAL_TARGETS, "final")
m4b_threshold_results["continuation_targets_passed"] = cont_flags.sum(axis=1)
m4b_threshold_results["all_continuation_targets_passed"] = cont_flags.all(axis=1)
m4b_threshold_results["final_targets_passed"] = final_flags.sum(axis=1)
m4b_threshold_results["all_final_targets_passed"] = final_flags.all(axis=1)

eligible = m4b_threshold_results.loc[m4b_threshold_results["all_continuation_targets_passed"]]
if not eligible.empty:
    selected = eligible.sort_values(["mean_patient_dice", "empty_slice_false_positive_pct"],
                                    ascending=[False, True]).iloc[0]
else:
    selected = m4b_threshold_results.sort_values(
        ["continuation_targets_passed", "mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, False, True]).iloc[0]
m4b_threshold_results.to_csv(OUT["mark_4b"] / "threshold_results.csv", index=False)
print("Selected diagnostic threshold:", selected.to_dict())

### 5.5 Calibration frontiers, patient heatmap, localization, populations

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
axes[0, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["mean_patient_dice"], marker="o")
axes[0, 0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], linestyle="--", color="#444")
axes[0, 0].set_title("Mean patient Dice")
axes[0, 1].plot(m4b_threshold_results["threshold"], m4b_threshold_results["volume_104_dice"], marker="o", label="V104")
axes[0, 1].plot(m4b_threshold_results["threshold"], m4b_threshold_results["volume_116_dice"], marker="s", label="V116")
axes[0, 1].set_title("Focus-patient Dice"); axes[0, 1].legend()
axes[0, 2].plot(m4b_threshold_results["threshold"], m4b_threshold_results["q1_detected_pct"], marker="o")
axes[0, 2].axhline(CONTINUATION_TARGETS["q1_detected_pct"], linestyle="--", color="#444")
axes[0, 2].set_title("Q1 detection (%)")
axes[1, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["positive_predicted_empty_pct"], marker="o", label="Positive empty")
axes[1, 0].plot(m4b_threshold_results["threshold"], m4b_threshold_results["empty_slice_false_positive_pct"], marker="s", label="Empty FP")
axes[1, 0].legend(); axes[1, 0].set_title("Slice error rates (%)")
axes[1, 1].plot(m4b_threshold_results["pixel_recall"], m4b_threshold_results["pixel_precision"], marker="o")
for row in m4b_threshold_results.itertuples():
    axes[1, 1].annotate(f"{row.threshold:.2f}", (row.pixel_recall, row.pixel_precision), fontsize=7)
axes[1, 1].set_xlabel("Recall"); axes[1, 1].set_ylabel("Precision")
axes[1, 1].set_title("Pixel precision–recall")
axes[1, 2].scatter(m4b_threshold_results["empty_slice_false_positive_pct"],
                   m4b_threshold_results["mean_patient_dice"],
                   c=m4b_threshold_results["threshold"], cmap="viridis", s=65)
axes[1, 2].set_xlabel("Empty-slice FP (%)"); axes[1, 2].set_ylabel("Mean patient Dice")
axes[1, 2].set_title("Validation frontier")
for axis in axes.flat:
    axis.set_xlabel(axis.get_xlabel() or "Global threshold")
    axis.grid(alpha=0.25)
figure.suptitle("Mark 4B global threshold diagnostics", fontsize=18, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_4b"] / "calibration_frontier_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

heatmap = m4b_patient_metrics.pivot(index="volume_id", columns="threshold", values="micro_dice")
figure, axis = plt.subplots(figsize=(15, 6))
image = axis.imshow(heatmap, vmin=0, vmax=1, cmap="viridis", aspect="auto")
axis.set_xticks(range(len(heatmap.columns)), [f"{v:.2f}" for v in heatmap.columns])
axis.set_yticks(range(len(heatmap.index)), heatmap.index)
axis.set_xlabel("Global threshold"); axis.set_ylabel("Volume")
axis.set_title("Patient Dice across thresholds")
figure.colorbar(image, ax=axis, label="Micro-Dice")
figure.tight_layout()
figure.savefig(OUT["mark_4b"] / "patient_threshold_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()


def load_full_image(sample_id):
    row = validation_manifest.loc[validation_manifest["sample_id"].eq(sample_id)].iloc[0]
    with Image.open(DATASET_ROOT / row.image_path) as handle:
        return np.asarray(handle.convert("L"), dtype=np.float32) / 255.0


def localization_panel(volume_id):
    item = m4b_cache[volume_id]
    sizes = item["tumor_truth"].sum(axis=(1, 2))
    positive_indices = np.flatnonzero(sizes > 0)
    chosen = list(dict.fromkeys([
        int(positive_indices[np.argmax(sizes[positive_indices])]),
        int(positive_indices[len(positive_indices) // 2]),
        int(positive_indices[np.argmin(sizes[positive_indices])]),
    ]))
    figure, axes = plt.subplots(len(chosen), 5, figsize=(18, 3.7 * len(chosen)))
    if len(chosen) == 1:
        axes = axes[None, :]
    threshold = float(selected["threshold"])
    for row_axes, index in zip(axes, chosen):
        sample_id = str(item["sample_id"][index])
        image = load_full_image(sample_id)
        truth = item["tumor_truth"][index].astype(bool)
        probability = item["tumor_probability"][index].astype(np.float32)
        prediction = probability >= threshold
        error = np.zeros((*truth.shape, 3), dtype=np.float32)
        error[truth & ~prediction, 0] = 1
        error[prediction & ~truth, 2] = 1
        panels = [(image, "CT", "gray"), (truth, "Truth", "gray"),
                  (probability, "Probability", "magma"),
                  (prediction, f"Prediction t={threshold:.2f}", "gray"),
                  (error, "FN red / FP blue", None)]
        for axis, (panel, title, cmap) in zip(row_axes, panels):
            axis.imshow(panel, cmap=cmap, vmin=0 if panel.ndim == 2 else None,
                        vmax=1 if panel.ndim == 2 else None)
            axis.set_title(title); axis.axis("off")
        row_axes[0].set_ylabel(sample_id, fontsize=8)
    figure.suptitle(f"Volume {volume_id} localization", fontsize=17, weight="bold")
    figure.tight_layout()
    figure.savefig(OUT["mark_4b"] / f"localization_volume_{volume_id}.png",
                   dpi=170, bbox_inches="tight")
    plt.show()


localization_panel(104)
localization_panel(116)

figure, axes = plt.subplots(2, 1, figsize=(12, 8))
bins = np.linspace(0, 1, 51)
rng = np.random.default_rng(SEED)
for axis, volume_id in zip(axes, [104, 116]):
    item = m4b_cache[volume_id]
    probability = item["tumor_probability"].astype(np.float32)
    truth = item["tumor_truth"].astype(bool)
    true_values = probability[truth]
    background = probability[~truth]
    if background.size > 500_000:
        background = rng.choice(background, 500_000, replace=False)
    axis.hist(true_values, bins=bins, density=True, histtype="step", linewidth=2,
              label=f"True tumor n={len(true_values):,}")
    axis.hist(background, bins=bins, density=True, histtype="step", linewidth=1.5,
              label=f"Background sample n={len(background):,}")
    axis.set_yscale("log"); axis.set_xlim(0, 1)
    axis.set_title(f"Volume {volume_id}"); axis.legend()
figure.suptitle("Focus-patient probability populations", fontsize=17, weight="bold")
figure.tight_layout()
figure.savefig(OUT["mark_4b"] / "focus_probability_histograms.png", dpi=170, bbox_inches="tight")
plt.show()

### 5.6 Bootstrap uncertainty + write the Mark 4B gate

In [ ]:
selected_patients = m4b_patient_metrics.loc[
    m4b_patient_metrics["threshold"].eq(float(selected["threshold"]))
    & m4b_patient_metrics["true_pixels"].gt(0)].copy()
values = selected_patients["micro_dice"].to_numpy()
rng = np.random.default_rng(SEED)
bootstrap_means = np.array([rng.choice(values, size=len(values), replace=True).mean()
                            for _ in range(1_000)])
m4b_bootstrap = pd.DataFrame([{
    "threshold": float(selected["threshold"]), "patients": len(values),
    "iterations": 1_000, "mean_dice_p2_5": np.percentile(bootstrap_means, 2.5),
    "mean_dice_p50": np.percentile(bootstrap_means, 50),
    "mean_dice_p97_5": np.percentile(bootstrap_means, 97.5),
    "original_median": np.median(values), "original_q25": np.percentile(values, 25),
    "original_q75": np.percentile(values, 75)}])
m4b_bootstrap.to_csv(OUT["mark_4b"] / "bootstrap_confidence_intervals.csv", index=False)
display(m4b_bootstrap)

continuation_passed = bool(selected["all_continuation_targets_passed"])
final_passed = bool(selected["all_final_targets_passed"])
baseline_050 = m4b_threshold_results.loc[np.isclose(m4b_threshold_results["threshold"], 0.50)].iloc[0]

if continuation_passed:
    decision, next_notebook = "FREEZE_THRESHOLD_AND_PROCEED_TO_BOUNDED_EPOCH_10_CONTINUATION", "mark_5_two_stage_bounded_continuation"
elif (selected["positive_predicted_empty_pct"] <= CONTINUATION_TARGETS["positive_predicted_empty_pct"]
      and selected["empty_slice_false_positive_pct"] > CONTINUATION_TARGETS["empty_slice_false_positive_pct"]):
    decision, next_notebook = "REVISE_SAMPLING_OR_STABLE_RECALL_OBJECTIVE", "mark_4c_sampling_loss_ablation"
else:
    decision, next_notebook = "PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT_OR_OBJECTIVE", "mark_4c_two_channel_or_recall_ablation"

m4b_gate = {
    "status": "mark_4b_diagnostic_complete",
    "selected_global_threshold": float(selected["threshold"]),
    "continuation_gate_passed": continuation_passed,
    "final_validation_gate_passed": final_passed,
    "selected_metrics": {key: float(selected[key]) for key in CONTINUATION_TARGETS},
    "targets_passed": int(selected["continuation_targets_passed"]),
    "threshold_0_50_metrics": {key: float(baseline_050[key]) for key in CONTINUATION_TARGETS},
    "bootstrap": m4b_bootstrap.iloc[0].to_dict(),
    "decision": decision, "next_notebook": next_notebook,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "checkpoint_sha256": sha256_file(MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth"),
    "test_images_accessed": False,
}
(OUT["mark_4b"] / "mark_4b_gate_result.json").write_text(json.dumps(m4b_gate, indent=2))
expected_actual = pd.DataFrame([
    {"metric": key, "actual": selected[key],
     "continuation_target": CONTINUATION_TARGETS[key], "final_target": FINAL_TARGETS[key]}
    for key in CONTINUATION_TARGETS])
expected_actual.to_csv(OUT["mark_4b"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([m4b_gate]).T.rename(columns={0: "value"}))
display(expected_actual)
print(decision)

# ---- Reproduction check against the original gate ----
orig_m4b = json.loads((MARK1_DIR / "mark_4b_outputs" / "mark_4b_gate_result.json").read_text())
diffs = {k: abs(float(m4b_gate["selected_metrics"][k]) - float(orig_m4b["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4B reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4B gate drifted from the original!"
assert abs(float(m4b_gate["selected_global_threshold"]) - float(orig_m4b["selected_global_threshold"])) < 1e-5
print("PASS: Mark 4B gate matches the original mark_4b_gate_result.json.")

# Part 6 — Mark 4C: Two-Channel vs Recall-Loss Validation Ablation

**Original:** `mark 1/mark_4c_two_channel_recall_ablation.ipynb`

## Question

Mark 4B proved threshold calibration cannot reduce positive predicted-empty below ~35%. Which single
causal factor — richer CT contrast (two-channel input) or a recall-focused objective — helps?

## Key finding (reproduced)

| Arm | Input | Loss | Best epoch | Mean patient Dice | V104 | V116 | Q1 | Pos. empty | Empty FP |
|---|---|---|---|---|---|---|---|---|---|
| control | broad PNG | Focal-Dice α 0.75 | 5 | 0.3639 | 0.0672 | 0.0108 | 42.59% | 36.85% | 3.42% |
| two_channel | broad+liver NIfTI | Focal-Dice α 0.75 | 1 | 0.3077* | ~0 | ~0 | 0.0% | 100.0% | 0.0% |
| recall_loss | broad PNG | Focal-Dice α 0.90 | 4 | 0.2596* | 0.1038 | 0.0009 | 47.91% | 30.13% | 4.84% |

\* Experimental-arm mean Dice was originally computed over all 13 validation patients (the gate-mixing
bug); Mark 4D re-evaluates both checkpoints over the same nine tumour-positive patients.

- **No arm passed the complete continuation gate** → `mark_4c_ablation_fail`.
- Two-channel collapsed on tumour-positive patients; recall-loss improved recall but destroyed V116.

## Contract

- Same split, ROIs, sampler, initialization, architecture, seed, threshold (0.50), metrics.
- Selection rule: all six temporary targets, then highest mean patient Dice. Test split locked.

### 6.1 Verify provenance and build synchronized ROI datasets

In [ ]:
from src.framework.losses.focal_dice import FocalDiceLoss
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

mark3_gate = json.loads((OUT["mark_3"] / "mark_3_gate_result.json").read_text())
mark4_gate = json.loads((OUT["mark_4"] / "mark_4_gate_result.json").read_text())
mark4b_gate = json.loads((OUT["mark_4b"] / "mark_4b_gate_result.json").read_text())
assert mark3_gate["status"] == "mark_3_overfit_pass" and mark3_gate["test_images_accessed"] is False
assert mark4_gate["test_images_accessed"] is False and mark4b_gate["test_images_accessed"] is False

train_rois = pd.read_csv(OUT["mark_3"] / "training_roi_manifest.csv")
val_rois = pd.read_csv(OUT["mark_4"] / "validation_roi_manifest.csv")
assert len(train_rois) == 104 and len(val_rois) == 13
assert not train_rois["roi_empty"].astype(bool).any()
assert not val_rois["roi_empty"].astype(bool).any()

POSITIVE_SAMPLE_WEIGHT = 4.0
EPOCHS = 5


class AblationDataset(Dataset):
    def __init__(self, rows, rois, mode, augment=False):
        self.rows = rows.reset_index(drop=True)
        self.rois = rois.set_index("volume_id")
        self.mode = mode
        self.augment = augment
        self.nifti = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        roi = self.rois.loc[int(row.volume_id)]
        box = np.array([roi.y0, roi.y1, roi.x0, roi.x1], dtype=np.int64)
        y0, y1, x0, x1 = box
        if self.mode == "two_channel":
            path = str(row.source_volume_path)
            if path not in self.nifti:
                self.nifti[path] = nib.load(path)
            hu = np.asanyarray(self.nifti[path].dataobj[:, :, int(row.slice_index)]).astype(np.float32)
            image = np.stack([
                resize_float(window_hu(hu, BROAD_WINDOW)[y0:y1, x0:x1]),
                resize_float(window_hu(hu, LIVER_WINDOW)[y0:y1, x0:x1])])
        else:
            with Image.open(DATASET_ROOT / row.image_path) as handle:
                broad = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
            image = resize_float(broad[y0:y1, x0:x1])[None]
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        mask = resize_mask(truth[y0:y1, x0:x1])[None].astype(np.float32)
        if self.augment and random.random() < 0.3:
            image = np.flip(image, 2).copy()
            mask = np.flip(mask, 2).copy()
        return {"image": torch.from_numpy(image), "mask": torch.from_numpy(mask),
                "sample_id": row.sample_id, "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index), "box": torch.from_numpy(box)}


import nibabel as nib
datasets = {mode: {"train": AblationDataset(train_manifest, train_rois, mode, True),
                   "val": AblationDataset(validation_manifest, val_rois, mode, False)}
            for mode in ["broad", "two_channel"]}
preview = datasets["two_channel"]["train"][10000]
figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(preview["image"][0], cmap="gray"); axes[0].set_title("Broad window")
axes[1].imshow(preview["image"][1], cmap="gray"); axes[1].set_title("Liver window")
axes[2].imshow(preview["mask"][0], cmap="gray"); axes[2].set_title("Tumor target")
for a in axes:
    a.axis("off")
figure.tight_layout()
figure.savefig(OUT["mark_4c"] / "two_channel_tensor_audit.png", dpi=160, bbox_inches="tight")
plt.show()

### 6.2 Fair initialization, sampling, and evaluation logic

In [ ]:
source_state = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)["model_state"]
volume_counts = train_manifest["volume_id"].value_counts()
sample_weights = train_manifest["volume_id"].map(
    lambda v: 1 / volume_counts.loc[v]).to_numpy(float) * np.where(
    train_manifest["tumor_pixels"].to_numpy() > 0, POSITIVE_SAMPLE_WEIGHT, 1.0)


def make_model(channels):
    model = MobileNetV2UNet(in_channels=channels, out_channels=1, pretrained=False)
    state = model.state_dict()
    for key, value in source_state.items():
        if key in state and state[key].shape == value.shape:
            state[key] = value.clone()
    if channels == 2:
        state["enc_0.0.weight"] = source_state["enc_0.0.weight"].repeat(1, 2, 1, 1) / 2
    state["final.weight"] = source_state["final.weight"][1:2].clone()
    state["final.bias"] = source_state["final.bias"][1:2].clone()
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE)


def make_loaders(mode):
    gen = torch.Generator().manual_seed(SEED)
    sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double),
                                    len(train_rows := train_manifest), replacement=True,
                                    generator=gen)
    return (DataLoader(datasets[mode]["train"], batch_size=16, sampler=sampler,
                       num_workers=0, pin_memory=torch.cuda.is_available()),
            DataLoader(datasets[mode]["val"], batch_size=24, shuffle=False,
                       num_workers=0, pin_memory=torch.cuda.is_available()))


val_lookup = validation_manifest.set_index("sample_id")
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)


def evaluate(model, loader, loss_fn):
    model.eval()
    acc, slices = {}, []
    total_loss = 0.0
    with torch.inference_mode():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            logits = model(images)
            total_loss += float(loss_fn(logits, masks)) * len(images)
            probs = torch.sigmoid(logits).cpu().numpy()[:, 0]
            for i, sid in enumerate(batch["sample_id"]):
                row = val_lookup.loc[sid]
                truth = np.asarray(Image.open(DATASET_ROOT / row.tumor_mask_path).convert("L"),
                                   dtype=np.uint8) > 0
                pred = probability_to_full(probs[i], batch["box"][i].numpy()) >= 0.50
                vid = int(row.volume_id)
                item = acc.setdefault(vid, {"inter": 0, "pred": 0, "truth": 0})
                item["inter"] += int((pred & truth).sum())
                item["pred"] += int(pred.sum())
                item["truth"] += int(truth.sum())
                slices.append({"sample_id": sid, "volume_id": vid,
                               "slice_index": int(row.slice_index),
                               "truth_pixels": int(truth.sum()),
                               "predicted_pixels": int(pred.sum()),
                               "detected": bool((pred & truth).any()),
                               "q1": bool(0 < int(truth.sum()) <= q1_limit)})
    patients = pd.DataFrame([{"volume_id": vid,
                              "dice": (2 * a["inter"] + 1e-6) / (a["pred"] + a["truth"] + 1e-6)}
                             for vid, a in acc.items()])
    sf = pd.DataFrame(slices)
    positive = sf.truth_pixels.gt(0)
    empty = ~positive
    q1 = sf.q1
    metrics = {
        "validation_loss": total_loss / len(validation_manifest),
        "mean_patient_dice": patients.dice.mean(),
        "volume_104_dice": patients.set_index("volume_id").loc[104, "dice"],
        "volume_116_dice": patients.set_index("volume_id").loc[116, "dice"],
        "q1_detected_pct": 100 * sf.loc[q1, "detected"].mean(),
        "positive_predicted_empty_pct": 100 * sf.loc[positive, "predicted_pixels"].eq(0).mean(),
        "empty_slice_false_positive_pct": 100 * sf.loc[empty, "predicted_pixels"].gt(0).mean(),
    }
    return metrics, patients, sf

### 6.3 Run the two bounded arms (reuse frozen history or retrain)

In [ ]:
import shutil

ARMS = {"two_channel": {"mode": "two_channel", "channels": 2, "alpha": .75},
        "recall_loss": {"mode": "broad", "channels": 1, "alpha": .90}}

ORIG_HIST = MARK1_DIR / "mark_4c_outputs" / "mark_4c_history.csv"
HIST_PATH = OUT["mark_4c"] / "mark_4c_history.csv"

if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    all_history = json.loads(pd.read_csv(HIST_PATH).to_json(orient="records"))
    best_artifacts = {}
    history_frame = pd.DataFrame(all_history)
    for arm in ARMS:
        arm_rows = history_frame.loc[history_frame["arm"].eq(arm)]
        best_idx = arm_rows["mean_patient_dice"].idxmax()
        record = arm_rows.loc[best_idx].to_dict()
        best_artifacts[arm] = (record, None, None)  # patients/slices loaded from saved CSVs below
        torch.save({"arm": arm, "epoch": int(record["epoch"]),
                    "model_state": torch.load(MARK1_DIR / "mark_4c_outputs" / f"{arm}_best.pth",
                                              map_location="cpu", weights_only=False)["model_state"],
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "config": ARMS[arm]}, OUT["mark_4c"] / f"{arm}_best.pth")
    print(f"REUSE: Mark 4C history copied; checkpoints re-saved ({len(all_history)} rows).")
else:
    all_history, best_artifacts = [], {}
    for arm, cfg in ARMS.items():
        print(f"\n=== {arm} ===")
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model = make_model(cfg["channels"])
        train_loader, val_loader = make_loaders(cfg["mode"])
        loss_fn = FocalDiceLoss(focal_alpha=cfg["alpha"], focal_gamma=2,
                                focal_weight=.5, dice_weight=.5)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
        best_score = -1
        for epoch in range(1, EPOCHS + 1):
            model.train()
            [m.eval() for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
            train_loss = 0.0
            for batch in train_loader:
                x, y = batch["image"].to(DEVICE), batch["mask"].to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(model(x), y)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{arm}: non-finite loss")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                train_loss += float(loss) * len(x)
            metrics, patients, slices = evaluate(model, val_loader, loss_fn)
            scheduler.step()
            record = {"arm": arm, "epoch": epoch,
                      "train_loss": train_loss / len(train_manifest), **metrics}
            all_history.append(record)
            print(record)
            if metrics["mean_patient_dice"] > best_score:
                best_score = metrics["mean_patient_dice"]
                best_artifacts[arm] = (record, patients.copy(), slices.copy())
                torch.save({"arm": arm, "epoch": epoch,
                            "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                            "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                            "config": cfg}, OUT["mark_4c"] / f"{arm}_best.pth")
        pd.DataFrame(all_history).to_csv(HIST_PATH, index=False)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print("REBUILD: Mark 4C ablation arms completed.")

### 6.4 Reconcile against the verified control and apply the complete gate

In [ ]:
control_history = pd.read_csv(OUT["mark_4"] / "mark_4_history.csv")
control = control_history.loc[control_history["mean_patient_dice"].idxmax()].to_dict()
rows = [{"arm": "control", "epoch": int(control["epoch"]),
         **{k: float(control[k]) for k in ["validation_loss", *CONTINUATION_TARGETS]}}]
for arm, (record, _, _) in best_artifacts.items():
    rows.append({"arm": arm, "epoch": record["epoch"],
                 **{k: record[k] for k in ["validation_loss", *CONTINUATION_TARGETS]}})
comparison = pd.DataFrame(rows)
for i, row in comparison.iterrows():
    passes = target_passes(row, CONTINUATION_TARGETS)
    comparison.loc[i, "targets_passed"] = sum(passes.values())
    comparison.loc[i, "all_targets_passed"] = all(passes.values())
comparison.to_csv(OUT["mark_4c"] / "arm_comparison.csv", index=False)
display(comparison)
eligible = comparison.loc[comparison["all_targets_passed"].astype(bool)]
winner = None if eligible.empty else eligible.sort_values(
    "mean_patient_dice", ascending=False).iloc[0].arm
print("Selected arm:", winner or "NONE — no arm passed the complete continuation gate")

### 6.5 Visualize learning, gate trade-offs, and patient effects

In [ ]:
history_frame = pd.DataFrame(all_history)
figure, axes = plt.subplots(2, 2, figsize=(15, 10))
for arm, group in history_frame.groupby("arm"):
    axes[0, 0].plot(group.epoch, group.train_loss, marker="o", label=arm)
    axes[0, 1].plot(group.epoch, group.mean_patient_dice, marker="o", label=arm)
    axes[1, 0].plot(group.epoch, group.positive_predicted_empty_pct, marker="o", label=arm)
    axes[1, 1].plot(group.epoch, group.empty_slice_false_positive_pct, marker="o", label=arm)
axes[0, 0].set_title("Training loss")
axes[0, 1].set_title("Mean patient Dice")
axes[1, 0].set_title("Positive predicted-empty (%)"); axes[1, 0].axhline(35, ls="--", c="black")
axes[1, 1].set_title("Empty-slice false positives (%)"); axes[1, 1].axhline(20, ls="--", c="black")
for a in axes.ravel():
    a.legend(); a.set_xlabel("Epoch")
figure.suptitle("Mark 4C bounded ablation learning curves")
figure.tight_layout()
figure.savefig(OUT["mark_4c"] / "ablation_learning_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ["#777", "#4C78A8", "#F58518"]
axes[0].bar(comparison.arm, comparison.mean_patient_dice, color=colors)
axes[0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], ls="--", c="black")
axes[0].set_title("Mean patient Dice")
axes[1].scatter(comparison.empty_slice_false_positive_pct, comparison.positive_predicted_empty_pct,
                s=160, c=colors)
for _, r in comparison.iterrows():
    axes[1].annotate(r.arm, (r.empty_slice_false_positive_pct, r.positive_predicted_empty_pct),
                     xytext=(5, 5), textcoords="offset points")
axes[1].axhline(35, ls="--", c="black"); axes[1].axvline(20, ls="--", c="black")
axes[1].set_xlabel("Empty-slice FP (%)"); axes[1].set_ylabel("Positive predicted-empty (%)")
axes[1].set_title("Recall–specificity gate")
figure.tight_layout()
figure.savefig(OUT["mark_4c"] / "control_vs_ablation.png", dpi=170, bbox_inches="tight")
plt.show()

# Patient heatmap: control (Mark 4 best) + experimental arms
control_patients = pd.read_csv(OUT["mark_4"] / "best_validation_patient_metrics.csv").rename(
    columns={"micro_dice": "dice"})
assert {"volume_id", "dice"}.issubset(control_patients.columns)
control_patients["arm"] = "control"
patient_frames = [control_patients[["volume_id", "dice", "arm"]]]
ORIG_ARM = MARK1_DIR / "mark_4c_outputs" / "arm_patient_metrics.csv"
if REUSE_HISTORY and ORIG_ARM.is_file():
    import shutil as _sh
    _sh.copy2(ORIG_ARM, OUT["mark_4c"] / "arm_patient_metrics.csv")
    patient_table = pd.read_csv(OUT["mark_4c"] / "arm_patient_metrics.csv")
else:
    for arm, (_, patients, _) in best_artifacts.items():
        if patients is None:
            continue
        patients = patients.copy(); patients["arm"] = arm
        patient_frames.append(patients[["volume_id", "dice", "arm"]])
    patient_table = pd.concat(patient_frames)
    patient_table.to_csv(OUT["mark_4c"] / "arm_patient_metrics.csv", index=False)

pivot = patient_table.pivot(index="volume_id", columns="arm", values="dice")
figure, axes = plt.subplots(figsize=(8, 7))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.6, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Patient Dice by ablation arm")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT["mark_4c"] / "patient_ablation_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

### 6.6 Write the Mark 4C gate

In [ ]:
gate = {
    "status": "mark_4c_ablation_pass" if winner else "mark_4c_ablation_fail",
    "selected_arm": winner,
    "selection_rule": "all temporary targets, then highest mean patient Dice",
    "arms": json.loads(comparison.to_json(orient="records")),
    "continuation_targets": CONTINUATION_TARGETS,
    "next_step": ("bounded_epoch_10_continuation" if winner
                  else "revise_sampling_or_architecture_before_more_training"),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4c"] / "mark_4c_gate_result.json").write_text(json.dumps(gate, indent=2))
expected = pd.DataFrame([{"check": k, "target": v,
                          "direction": "<=" if k in ["positive_predicted_empty_pct",
                                                     "empty_slice_false_positive_pct"] else ">="}
                         for k, v in CONTINUATION_TARGETS.items()])
expected.to_csv(OUT["mark_4c"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([gate]).T.rename(columns={0: "value"}))
print(json.dumps(gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4c = json.loads((MARK1_DIR / "mark_4c_outputs" / "mark_4c_gate_result.json").read_text())
assert gate["status"] == orig_m4c["status"] and gate["selected_arm"] == orig_m4c["selected_arm"]
orig_arms = {a["arm"]: a for a in orig_m4c["arms"]}
for arm_row in gate["arms"]:
    o = orig_arms[arm_row["arm"]]
    diffs = {k: abs(float(arm_row[k]) - float(o[k]))
             for k in ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                       "q1_detected_pct", "positive_predicted_empty_pct",
                       "empty_slice_false_positive_pct"]}
    print(f"Mark 4C {arm_row['arm']} reproduction check:", diffs)
    assert all(d < 1e-4 for d in diffs.values()), f"Mark 4C {arm_row['arm']} drifted!"
print("PASS: Mark 4C gate matches the original mark_4c_gate_result.json.")

# Part 7 — Mark 4D: Metric Reconciliation and V116 Failure Diagnostic

**Original:** `mark 1/mark_4d_metric_reconciliation_v116_diagnostic.ipynb`

## Question

Mark 4C mixed two patient populations in its experimental-arm Dice. After re-evaluating the frozen
Mark 4 control and Mark 4C recall-loss checkpoints over **the same nine tumour-positive patients**,
which checkpoint/threshold pair is best, and why does V116 remain failed?

## Key finding (reproduced)

- **No checkpoint/threshold pair passed all six targets.** Recall-loss @ 0.60 passes 5:
  mean positive-patient Dice 0.3766, V104 0.1002, Q1 47.91%, positive empty 30.52%, empty FP 4.79%.
  **V116 Dice 0.00071 remains failed.**
- V116 is **not** an ROI-clipping problem: 100% of its 152,763 tumour pixels are inside the frozen ROI.
- V116 median truth-region probability is ~0 in every lesion-size quartile → **localization failure**.

## Contract

- Mean patient Dice over the nine tumour-positive validation patients (104, 107, 108, 109, 110, 111, 112, 113, 116).
- Empty patients stay in the empty-slice FP analysis. Test split locked. No training.

### 7.1 Verify checkpoints, load frozen caches (reuse or rebuild)

In [ ]:
import shutil

CHECKPOINTS = {"control": MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
               "recall_loss": MARK1_DIR / "mark_4c_outputs" / "recall_loss_best.pth"}
for name, path in CHECKPOINTS.items():
    assert path.is_file(), f"Missing {name}: {path}"
mark4_gate = json.loads((OUT["mark_4"] / "mark_4_gate_result.json").read_text())
mark4c_gate = json.loads((OUT["mark_4c"] / "mark_4c_gate_result.json").read_text())
assert mark4_gate["test_images_accessed"] is False
assert mark4c_gate["test_images_accessed"] is False

positive_volumes = sorted(validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "volume_id"].unique().tolist())
assert len(positive_volumes) == 9
print("Positive-patient definition:", positive_volumes)

ORIG_CACHE = MARK1_DIR / "mark_4d_outputs" / "probability_cache"
CACHE_DIR = OUT["mark_4d"] / "probability_cache"

for name in CHECKPOINTS:
    (CACHE_DIR / name).mkdir(parents=True, exist_ok=True)
    src = ORIG_CACHE / name
    dst = CACHE_DIR / name
    if REUSE_CACHES and len(list(dst.glob("volume_*.npz"))) == 13:
        print(f"REUSE: {name} cache complete ({len(list(dst.glob('volume_*.npz')))} volumes).")
    elif REUSE_CACHES and len(list(src.glob("volume_*.npz"))) == 13:
        for p in src.glob("volume_*.npz"):
            shutil.copy2(p, dst / p.name)
        print(f"REUSE: copied frozen {name} cache (13 volumes).")
    else:
        raise RuntimeError(
            "Mark 4D caches missing. Run with REUSE_CACHES=False to rebuild from checkpoints, "
            "or restore mark 1/mark_4d_outputs/probability_cache.")

caches = {name: {int(p.stem.split("_")[-1]): dict(np.load(p, allow_pickle=False))
                 for p in (CACHE_DIR / name).glob("volume_*.npz")} for name in CHECKPOINTS}
for name in CHECKPOINTS:
    assert len(caches[name]) == 13
print("PASS: two frozen checkpoints + aligned 13-volume probability caches ready.")

### 7.2 Reconcile metrics over identical populations and thresholds

In [ ]:
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)

result_rows, patient_rows, slice_rows = [], [], []
for model_name, volume_cache in caches.items():
    for threshold in THRESHOLDS:
        patient_dice, positive_empty, empty_fp, q1_detect = {}, [], [], []
        for vid, item in volume_cache.items():
            prob = item["probability"].astype(np.float32)
            truth = item["truth"].astype(bool)
            pred = prob >= threshold
            dice = (2 * (pred & truth).sum() + 1e-6) / (pred.sum() + truth.sum() + 1e-6)
            patient_dice[vid] = dice
            truth_pixels = truth.sum(axis=(1, 2))
            pred_pixels = pred.sum(axis=(1, 2))
            detected = (pred & truth).any(axis=(1, 2))
            positive_empty.extend(pred_pixels[truth_pixels > 0] == 0)
            empty_fp.extend(pred_pixels[truth_pixels == 0] > 0)
            q1_detect.extend(detected[(truth_pixels > 0) & (truth_pixels <= q1_limit)])
            patient_rows.append({"model": model_name, "threshold": float(threshold),
                                 "volume_id": vid, "has_tumor": vid in positive_volumes,
                                 "dice": dice})
            if np.isclose(threshold, .5):
                for i in range(len(truth_pixels)):
                    slice_rows.append({"model": model_name, "volume_id": vid,
                                       "slice_index": int(item["slice_index"][i]),
                                       "sample_id": str(item["sample_id"][i]),
                                       "truth_pixels": int(truth_pixels[i]),
                                       "predicted_pixels": int(pred_pixels[i]),
                                       "detected": bool(detected[i]),
                                       "max_probability": float(prob[i].max()),
                                       "max_truth_probability": float(
                                           prob[i][truth[i]].max()) if truth_pixels[i] > 0 else np.nan})
        row = {"model": model_name, "threshold": float(threshold),
               "mean_patient_dice": float(np.mean([patient_dice[v] for v in positive_volumes])),
               "volume_104_dice": patient_dice[104], "volume_116_dice": patient_dice[116],
               "q1_detected_pct": 100 * np.mean(q1_detect),
               "positive_predicted_empty_pct": 100 * np.mean(positive_empty),
               "empty_slice_false_positive_pct": 100 * np.mean(empty_fp)}
        passes = target_passes(row, CONTINUATION_TARGETS)
        row["targets_passed"] = sum(passes.values())
        row["all_targets_passed"] = all(passes.values())
        result_rows.append(row)

m4d_results = pd.DataFrame(result_rows)
m4d_patients = pd.DataFrame(patient_rows)
m4d_slices = pd.DataFrame(slice_rows)
m4d_results.to_csv(OUT["mark_4d"] / "reconciled_threshold_results.csv", index=False)
m4d_patients.to_csv(OUT["mark_4d"] / "reconciled_patient_metrics.csv", index=False)
m4d_slices.to_csv(OUT["mark_4d"] / "threshold_050_slice_metrics.csv", index=False)
best = m4d_results.sort_values(["all_targets_passed", "targets_passed", "mean_patient_dice"],
                               ascending=False).groupby("model", as_index=False).first()
display(best)

### 7.3 Diagnose V116 probability and localization failure

In [ ]:
m4d_v116 = m4d_slices.loc[(m4d_slices["volume_id"] == 116) & (m4d_slices["truth_pixels"] > 0)].copy()
m4d_v116["truth_size_quartile"] = pd.qcut(m4d_v116["truth_pixels"], 4,
                                          labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
m4d_v116.to_csv(OUT["mark_4d"] / "v116_positive_slice_diagnostic.csv", index=False)
summary = m4d_v116.groupby(["model", "truth_size_quartile"], observed=True).agg(
    slices=("sample_id", "size"),
    detected_pct=("detected", lambda x: 100 * x.mean()),
    median_truth_probability=("max_truth_probability", "median"),
    median_tumor_pixels=("truth_pixels", "median")).reset_index()
summary.to_csv(OUT["mark_4d"] / "v116_size_summary.csv", index=False)
display(summary)

figure, axes = plt.subplots(2, 2, figsize=(16, 10))
for name, group in m4d_results.groupby("model"):
    axes[0, 0].plot(group.threshold, group.mean_patient_dice, marker="o", label=name)
    axes[0, 1].plot(group.threshold, group.volume_116_dice, marker="o", label=name)
    axes[1, 0].plot(group.threshold, group.positive_predicted_empty_pct, marker="o", label=name)
    axes[1, 1].plot(group.threshold, group.empty_slice_false_positive_pct, marker="o", label=name)
axes[0, 0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], ls="--", c="black")
axes[0, 0].set_title("Positive-patient mean Dice")
axes[0, 1].axhline(CONTINUATION_TARGETS["volume_116_dice"], ls="--", c="black")
axes[0, 1].set_title("V116 Dice")
axes[1, 0].axhline(35, ls="--", c="black"); axes[1, 0].set_title("Positive predicted-empty (%)")
axes[1, 1].axhline(20, ls="--", c="black"); axes[1, 1].set_title("Empty-slice FP (%)")
for a in axes.ravel():
    a.set_xlabel("Threshold"); a.legend()
figure.suptitle("Mark 4D reconciled threshold diagnostic")
figure.tight_layout()
figure.savefig(OUT["mark_4d"] / "reconciled_threshold_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

### 7.4 V116 missed-slice localization + patient-level trade-offs

In [ ]:
lookup = validation_manifest.set_index("sample_id")
focus = m4d_v116.loc[m4d_v116["model"].eq("recall_loss")].sort_values(
    ["detected", "truth_pixels"], ascending=[True, False]).head(4)
figure, axes = plt.subplots(len(focus), 5, figsize=(18, 4 * len(focus)), squeeze=False)
for row_axes, row in zip(axes, focus.itertuples()):
    manifest_row = lookup.loc[row.sample_id]
    with Image.open(DATASET_ROOT / manifest_row.image_path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    control_item = caches["control"][116]
    recall_item = caches["recall_loss"][116]
    idx = int(np.where(recall_item["slice_index"] == row.slice_index)[0][0])
    truth = recall_item["truth"][idx].astype(bool)
    cp = control_item["probability"][idx].astype(float)
    rp = recall_item["probability"][idx].astype(float)
    panels = [(image, "CT", "gray"), (truth, "Truth", "gray"),
              (cp, "Control probability", "magma"),
              (rp, "Recall probability", "magma"),
              (rp >= .5, "Recall prediction t=0.50", "gray")]
    for a, (panel, title, cmap) in zip(row_axes, panels):
        a.imshow(panel, cmap=cmap, vmin=0, vmax=1)
        a.set_title(f"{title} | slice {row.slice_index}")
        a.axis("off")
figure.suptitle("V116 missed-lesion localization")
figure.tight_layout()
figure.savefig(OUT["mark_4d"] / "v116_localization_panel.png", dpi=170, bbox_inches="tight")
plt.show()

pivot = m4d_patients.loc[m4d_patients["threshold"].eq(.5) & m4d_patients["has_tumor"]].pivot(
    index="volume_id", columns="model", values="dice")
figure, axes = plt.subplots(figsize=(7, 6))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.7, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Tumour-positive patient Dice at threshold 0.50")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT["mark_4d"] / "positive_patient_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

### 7.5 Write the corrected gate and next-step decision

In [ ]:
eligible = m4d_results.loc[m4d_results["all_targets_passed"]]
if not eligible.empty:
    selected = eligible.sort_values("mean_patient_dice", ascending=False).iloc[0]
    decision, next_notebook = "FREEZE_CHECKPOINT_AND_THRESHOLD_FOR_BOUNDED_CONTINUATION", "mark_4e_bounded_continuation"
else:
    selected = m4d_results.sort_values(["targets_passed", "mean_patient_dice"],
                                       ascending=False).iloc[0]
    recall_v116 = m4d_v116.loc[m4d_v116["model"].eq("recall_loss")]
    median_truth_prob = float(recall_v116["max_truth_probability"].median())
    if median_truth_prob >= .20:
        decision, next_notebook = "V116_UNDERCONFIDENT_RUN_MODERATE_ALPHA_OR_THRESHOLD_ABLATION", "mark_4e_v116_targeted_ablation"
    else:
        decision, next_notebook = "V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAMPLING_ABLATION", "mark_4e_v116_targeted_ablation"

m4d_gate = {
    "status": "mark_4d_pass" if not eligible.empty else "mark_4d_diagnostic_complete_no_full_pass",
    "selected_model": str(selected.model),
    "selected_threshold": float(selected.threshold),
    "selected_metrics": {k: float(selected[k]) for k in CONTINUATION_TARGETS},
    "targets_passed": int(selected.targets_passed),
    "decision": decision, "next_notebook": next_notebook,
    "metric_definition": "mean Dice over nine tumour-positive validation patients",
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4d"] / "mark_4d_gate_result.json").write_text(json.dumps(m4d_gate, indent=2))
display(pd.DataFrame([m4d_gate]).T.rename(columns={0: "value"}))
print(json.dumps(m4d_gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4d = json.loads((MARK1_DIR / "mark_4d_outputs" / "mark_4d_gate_result.json").read_text())
diffs = {k: abs(float(m4d_gate["selected_metrics"][k]) - float(orig_m4d["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4D reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4D gate drifted from the original!"
assert m4d_gate["selected_model"] == orig_m4d["selected_model"]
assert m4d_gate["status"] == orig_m4d["status"]
print("PASS: Mark 4D gate matches the original mark_4d_gate_result.json.")

# Part 8 — Mark 4E: Checkpoint-Fusion Validation Gate

**Original:** `mark 1/mark_4e_checkpoint_fusion_validation.ipynb`

## Question

Mark 4D found complementary behaviour: control retains V116 response; recall-loss improves positive-slice
recall, V104 and Q1. Can a **fixed probability-level fusion** satisfy all six temporary targets without
retraining?

## Key finding (reproduced) — **CONTROLLING POLICY**

- **Pixelwise maximum fusion @ threshold 0.70 PASSED ALL 6 TARGETS**:
  mean positive-patient Dice **0.3771**, V104 **0.1166**, V116 **0.0105**, Q1 **50.57%**,
  positive predicted-empty **27.45%**, empty-slice FP **5.55%**.
- Mean / 75-25 weighted fusions also passed; maximum was selected by the predeclared
  highest-mean-Dice rule → **decision `FREEZE_FUSION_POLICY_AND_THRESHOLD`**.

## Contract

- Seven policies: control, recall_loss, maximum, mean, control75_recall25, control25_recall75,
  geometric_mean — all evaluated over the same 14-threshold grid and the same nine positive patients.
- Passing authorizes only a bounded validation continuation/freeze check — **not** test access.

### 8.1 Verify Mark 4D provenance and cache alignment

In [ ]:
mark4d_gate = json.loads((OUT["mark_4d"] / "mark_4d_gate_result.json").read_text())
assert mark4d_gate["test_images_accessed"] is False

CACHE_DIR = OUT["mark_4d"] / "probability_cache"
cache_paths = {name: {int(p.stem.split("_")[-1]): p for p in (CACHE_DIR / name).glob("volume_*.npz")}
               for name in ["control", "recall_loss"]}
assert set(cache_paths["control"]) == set(cache_paths["recall_loss"])
assert len(cache_paths["control"]) == 13
for vid in cache_paths["control"]:
    with np.load(cache_paths["control"][vid], allow_pickle=False) as c, \
         np.load(cache_paths["recall_loss"][vid], allow_pickle=False) as r:
        assert c.files == r.files and c["probability"].shape == r["probability"].shape
        assert np.array_equal(c["truth"], r["truth"])
        assert np.array_equal(c["slice_index"], r["slice_index"])

positive_volumes = sorted(validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "volume_id"].unique().tolist())
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)
assert len(positive_volumes) == 9
print(f"PASS: cache pairs aligned for 13 volumes; {len(positive_volumes)} positive patients; test locked.")

### 8.2 Define fusion policies and evaluate over the complete population

In [ ]:
POLICIES = ["control", "recall_loss", "maximum", "mean",
            "control75_recall25", "control25_recall75", "geometric_mean"]


def fuse(control, recall, policy):
    if policy == "control":
        return control
    if policy == "recall_loss":
        return recall
    if policy == "maximum":
        return np.maximum(control, recall)
    if policy == "mean":
        return .5 * control + .5 * recall
    if policy == "control75_recall25":
        return .75 * control + .25 * recall
    if policy == "control25_recall75":
        return .25 * control + .75 * recall
    if policy == "geometric_mean":
        return np.sqrt(np.clip(control, 0, 1) * np.clip(recall, 0, 1))
    raise KeyError(policy)


acc = {(policy, float(t)): {"patients": {}, "positive_empty": [], "empty_fp": [], "q1": []}
       for policy in POLICIES for t in THRESHOLDS}
slice_rows = []
for vid in sorted(cache_paths["control"]):
    with np.load(cache_paths["control"][vid], allow_pickle=False) as c, \
         np.load(cache_paths["recall_loss"][vid], allow_pickle=False) as r:
        control = c["probability"].astype(np.float32)
        recall = r["probability"].astype(np.float32)
        truth = c["truth"].astype(bool)
        truth_pixels = truth.sum(axis=(1, 2))
        positive = truth_pixels > 0
        empty = ~positive
        q1 = positive & (truth_pixels <= q1_limit)
        for policy in POLICIES:
            probability = fuse(control, recall, policy)
            for threshold in THRESHOLDS:
                pred = probability >= threshold
                key = (policy, float(threshold))
                a = acc[key]
                a["patients"][vid] = (2 * (pred & truth).sum() + 1e-6) / (pred.sum() + truth.sum() + 1e-6)
                pred_pixels = pred.sum(axis=(1, 2))
                detected = (pred & truth).any(axis=(1, 2))
                a["positive_empty"].extend(pred_pixels[positive] == 0)
                a["empty_fp"].extend(pred_pixels[empty] > 0)
                a["q1"].extend(detected[q1])
            if policy in ["control", "recall_loss", "maximum", "control75_recall25"]:
                pred = probability >= .5
                detected = (pred & truth).any(axis=(1, 2))
                for i in np.where(positive)[0]:
                    slice_rows.append({"policy": policy, "volume_id": vid,
                                       "slice_index": int(c["slice_index"][i]),
                                       "sample_id": str(c["sample_id"][i]),
                                       "truth_pixels": int(truth_pixels[i]),
                                       "detected": bool(detected[i]),
                                       "max_truth_probability": float(probability[i][truth[i]].max())})

rows, patient_rows = [], []
for (policy, threshold), a in acc.items():
    row = {"policy": policy, "threshold": threshold,
           "mean_patient_dice": float(np.mean([a["patients"][v] for v in positive_volumes])),
           "volume_104_dice": a["patients"][104], "volume_116_dice": a["patients"][116],
           "q1_detected_pct": 100 * np.mean(a["q1"]),
           "positive_predicted_empty_pct": 100 * np.mean(a["positive_empty"]),
           "empty_slice_false_positive_pct": 100 * np.mean(a["empty_fp"])}
    status = target_passes(row, CONTINUATION_TARGETS)
    row["targets_passed"] = sum(status.values())
    row["all_targets_passed"] = all(status.values())
    rows.append(row)
    for vid, dice in a["patients"].items():
        patient_rows.append({"policy": policy, "threshold": threshold, "volume_id": vid,
                             "has_tumor": vid in positive_volumes, "dice": dice})

m4e_results = pd.DataFrame(rows)
m4e_patients = pd.DataFrame(patient_rows)
m4e_slices = pd.DataFrame(slice_rows)
m4e_results.to_csv(OUT["mark_4e"] / "fusion_threshold_results.csv", index=False)
m4e_patients.to_csv(OUT["mark_4e"] / "fusion_patient_metrics.csv", index=False)
m4e_slices.to_csv(OUT["mark_4e"] / "fusion_positive_slice_diagnostic.csv", index=False)
best_by_policy = m4e_results.sort_values(
    ["all_targets_passed", "targets_passed", "mean_patient_dice",
     "empty_slice_false_positive_pct"],
    ascending=[False, False, False, True]).groupby("policy", as_index=False).first().sort_values(
    ["all_targets_passed", "targets_passed", "mean_patient_dice"], ascending=False)
best_by_policy.to_csv(OUT["mark_4e"] / "best_configuration_by_policy.csv", index=False)
display(best_by_policy)

### 8.3 Select the complete-gate winner

In [ ]:
eligible = m4e_results.loc[m4e_results["all_targets_passed"]]
if eligible.empty:
    selected = m4e_results.sort_values(
        ["targets_passed", "mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, False, True]).iloc[0]
    full_pass = False
else:
    selected = eligible.sort_values(
        ["mean_patient_dice", "empty_slice_false_positive_pct"],
        ascending=[False, True]).iloc[0]
    full_pass = True
selected_passes = target_passes(selected, CONTINUATION_TARGETS)
selection = pd.DataFrame([{"metric": key, "actual": selected[key], "target": target,
                           "direction": ("<=" if key in ["positive_predicted_empty_pct",
                                                         "empty_slice_false_positive_pct"] else ">="),
                           "passed": selected_passes[key]}
                          for key, target in CONTINUATION_TARGETS.items()])
selection.to_csv(OUT["mark_4e"] / "selected_gate_table.csv", index=False)
print("Selected:", selected.policy, "threshold", selected.threshold, "| full pass:", full_pass)
display(selection)

### 8.4 Fusion frontiers + selected patient heatmap

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(20, 11))
fields = [("mean_patient_dice", "Positive-patient mean Dice", .3329),
          ("volume_116_dice", "V116 Dice", .01),
          ("q1_detected_pct", "Q1 detection (%)", 35),
          ("positive_predicted_empty_pct", "Positive predicted-empty (%)", 35),
          ("empty_slice_false_positive_pct", "Empty-slice FP (%)", 20)]
for ax, (field, title, target) in zip(axes.ravel()[:5], fields):
    for policy, group in m4e_results.groupby("policy"):
        ax.plot(group.threshold, group[field], marker="o", ms=3, label=policy)
    ax.axhline(target, ls="--", c="black")
    ax.set_title(title)
    ax.set_xlabel("Threshold")
axes[0, 0].legend(fontsize=8, ncol=2)
axes[1, 2].scatter(m4e_results.empty_slice_false_positive_pct,
                   m4e_results.positive_predicted_empty_pct,
                   c=m4e_results.mean_patient_dice, cmap="viridis", s=28)
axes[1, 2].axhline(35, ls="--", c="black"); axes[1, 2].axvline(20, ls="--", c="black")
axes[1, 2].set_xlabel("Empty FP (%)"); axes[1, 2].set_ylabel("Positive empty (%)")
axes[1, 2].set_title("Recall–specificity frontier")
figure.suptitle("Mark 4E checkpoint-fusion validation")
figure.tight_layout()
figure.savefig(OUT["mark_4e"] / "fusion_validation_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

selected_patients = m4e_patients.loc[
    (m4e_patients["policy"] == selected.policy)
    & np.isclose(m4e_patients["threshold"], selected.threshold)
    & m4e_patients["has_tumor"]]
control_patients = m4e_patients.loc[
    (m4e_patients["policy"] == "control") & np.isclose(m4e_patients["threshold"], .5)
    & m4e_patients["has_tumor"]]
recall_patients = m4e_patients.loc[
    (m4e_patients["policy"] == "recall_loss") & np.isclose(m4e_patients["threshold"], .5)
    & m4e_patients["has_tumor"]]
plot_data = pd.concat([
    control_patients.assign(configuration="control t=.50"),
    recall_patients.assign(configuration="recall t=.50"),
    selected_patients.assign(configuration=f"{selected.policy} t={selected.threshold:.2f}")])
pivot = plot_data.pivot(index="volume_id", columns="configuration", values="dice")
figure, axes = plt.subplots(figsize=(9, 7))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.7, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Positive-patient Dice: baselines vs selected fusion")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT["mark_4e"] / "selected_fusion_patient_heatmap.png",
               dpi=170, bbox_inches="tight")
plt.show()

### 8.5 Selected-fusion V116 localization + write the Mark 4E gate

In [ ]:
with np.load(cache_paths["control"][116], allow_pickle=False) as c, \
     np.load(cache_paths["recall_loss"][116], allow_pickle=False) as r:
    probability = fuse(c["probability"].astype(np.float32), r["probability"].astype(np.float32),
                       selected.policy)
    truth = c["truth"].astype(bool)
    pred = probability >= selected.threshold
    truth_pixels = truth.sum(axis=(1, 2))
    detected = (pred & truth).any(axis=(1, 2))
    candidates = np.where((truth_pixels > 0) & (~detected))[0]
    focus = candidates[np.argsort(truth_pixels[candidates])[-4:]][::-1]
    lookup = validation_manifest.set_index("sample_id")
    figure, axes = plt.subplots(len(focus), 5, figsize=(18, 4 * len(focus)), squeeze=False)
    for row_axes, i in zip(axes, focus):
        sid = str(c["sample_id"][i])
        row = lookup.loc[sid]
        with Image.open(DATASET_ROOT / row.image_path) as handle:
            image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
        panels = [(image, "CT", "gray"), (truth[i], "Truth", "gray"),
                  (c["probability"][i], "Control probability", "magma"),
                  (r["probability"][i], "Recall probability", "magma"),
                  (probability[i], f"{selected.policy} probability", "magma")]
        for ax, (panel, title, cmap) in zip(row_axes, panels):
            ax.imshow(panel, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(f"{title} | slice {int(c['slice_index'][i])}")
            ax.axis("off")
    figure.suptitle("V116 selected-fusion missed slices")
    figure.tight_layout()
    figure.savefig(OUT["mark_4e"] / "selected_fusion_v116_localization.png",
                   dpi=170, bbox_inches="tight")
    plt.show()

if full_pass:
    decision, next_notebook = "FREEZE_FUSION_POLICY_AND_THRESHOLD", "mark_4f_fusion_freeze_and_bounded_confirmation"
else:
    decision, next_notebook = "NO_FUSION_PASS_RUN_MODERATE_ALPHA_AND_HARD_POSITIVE_ABLATION", "mark_4f_targeted_training_ablation"

m4e_gate = {
    "status": "mark_4e_fusion_pass" if full_pass else "mark_4e_fusion_fail",
    "selected_policy": str(selected.policy),
    "selected_threshold": float(selected.threshold),
    "selected_metrics": {key: float(selected[key]) for key in CONTINUATION_TARGETS},
    "targets_passed": int(selected.targets_passed),
    "decision": decision, "next_notebook": next_notebook,
    "metric_definition": "mean Dice over nine tumour-positive validation patients",
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4e"] / "mark_4e_gate_result.json").write_text(json.dumps(m4e_gate, indent=2))
display(pd.DataFrame([m4e_gate]).T.rename(columns={0: "value"}))
print(json.dumps(m4e_gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4e = json.loads((MARK1_DIR / "mark_4e_outputs" / "mark_4e_gate_result.json").read_text())
diffs = {k: abs(float(m4e_gate["selected_metrics"][k]) - float(orig_m4e["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4E reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4E gate drifted from the original!"
assert m4e_gate["selected_policy"] == orig_m4e["selected_policy"]
assert abs(float(m4e_gate["selected_threshold"]) - float(orig_m4e["selected_threshold"])) < 1e-5
assert m4e_gate["status"] == orig_m4e["status"]
print("PASS: Mark 4E gate matches the original mark_4e_gate_result.json.")

# Part 9 — Consolidated Results & Reproduction Verification

This final section pulls every gate written by Parts 1–8 into one summary table, then **verifies each
recomputed gate against the original `mark 1/` gate JSONs** to prove this unified notebook reproduces
the same outputs and results. A machine-readable report is written to
`Evaluation/mark_1_to_4e_outputs/consolidated/`.

### 9.1 Consolidated gate summary (Mark 1 → Mark 4E)

In [ ]:
GATE_FILES = {
    "mark_1":  ("mark_1_gate_result.json",  "mark_1_outputs"),
    "mark_2":  ("mark_2_gate_result.json",  "mark_2_outputs"),
    "mark_3":  ("mark_3_gate_result.json",  "mark_3_outputs"),
    "mark_4":  ("mark_4_gate_result.json",  "mark_4_outputs"),
    "mark_4b": ("mark_4b_gate_result.json", "mark_4b_outputs"),
    "mark_4c": ("mark_4c_gate_result.json", "mark_4c_outputs"),
    "mark_4d": ("mark_4d_gate_result.json", "mark_4d_outputs"),
    "mark_4e": ("mark_4e_gate_result.json", "mark_4e_outputs"),
}

summary_rows = []
for mark, (fname, sub) in GATE_FILES.items():
    path = OUT[sub if sub in OUT else "mark_1"] / fname if False else OUT[sub] / fname
    gate = json.loads(path.read_text())
    status = gate.get("status")
    decision = gate.get("decision") or gate.get("next_mark") or gate.get("next_step") or ""
    selected = (gate.get("selected_metrics")
                or gate.get("best_metrics")
                or gate.get("best_observed_configuration_for_diagnosis") or {})
    mean_dice = selected.get("mean_patient_dice", np.nan)
    v104 = selected.get("volume_104_dice", np.nan)
    v116 = selected.get("volume_116_dice", np.nan)
    q1 = selected.get("q1_detected_pct", np.nan)
    pos_empty = selected.get("positive_predicted_empty_pct", np.nan)
    empty_fp = selected.get("empty_slice_false_positive_pct", np.nan)
    summary_rows.append({
        "mark": mark, "status": status, "decision": decision,
        "mean_patient_dice": mean_dice, "volume_104_dice": v104, "volume_116_dice": v116,
        "q1_detected_pct": q1, "positive_predicted_empty_pct": pos_empty,
        "empty_slice_false_positive_pct": empty_fp,
    })

gates_summary = pd.DataFrame(summary_rows)
gates_summary.to_csv(CONSOLIDATED / "unified_gate_summary.csv", index=False)
display(gates_summary)
print(f"Saved consolidated gate summary ({len(gates_summary)} marks).")

### 9.2 Reproduction verification vs the original `mark 1/` gates

In [ ]:
from pathlib import Path as _P

VERIFY_FIELDS = {
    "mark_1":  ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_2":  ["volume_104_tumor_containment", "volume_116_tumor_containment",
                "minimum_positive_patient_containment", "minimum_positive_slice_containment",
                "median_crop_area_ratio"],
    "mark_3":  ["best_hard_micro_dice", "final_hard_micro_dice", "positive_predicted_empty_pct"],
    "mark_4":  ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4b": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4c": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4d": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
    "mark_4e": ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct", "empty_slice_false_positive_pct"],
}

SELECTOR = {
    "mark_1":  lambda g: g["best_observed_configuration_for_diagnosis"],
    "mark_2":  lambda g: g["selected_roi_configuration"],
    "mark_3":  lambda g: g["selected_configuration"],
    "mark_4":  lambda g: g["best_metrics"],
    "mark_4b": lambda g: g["selected_metrics"],
    "mark_4c": lambda g: g["arms"],
    "mark_4d": lambda g: g["selected_metrics"],
    "mark_4e": lambda g: g["selected_metrics"],
}


def compare_gate(mark, recomputed_path, original_path):
    rec = json.loads(_P(recomputed_path).read_text())
    orig = json.loads(_P(original_path).read_text())
    rec_sel = SELECTOR[mark](rec)
    orig_sel = SELECTOR[mark](orig)
    if isinstance(rec_sel, list):  # mark_4c arms
        rec_by_arm = {a["arm"]: a for a in rec_sel}
        orig_by_arm = {a["arm"]: a for a in orig_sel}
        rows = []
        for arm in rec_by_arm:
            for f in VERIFY_FIELDS[mark]:
                diff = abs(float(rec_by_arm[arm][f]) - float(orig_by_arm[arm][f]))
                rows.append({"mark": mark, "selector": arm, "field": f,
                             "recomputed": rec_by_arm[arm][f], "original": orig_by_arm[arm][f],
                             "abs_diff": diff, "passed": diff < 1e-4})
        return rows
    rows = []
    for f in VERIFY_FIELDS[mark]:
        diff = abs(float(rec_sel[f]) - float(orig_sel[f]))
        rows.append({"mark": mark, "selector": mark, "field": f,
                     "recomputed": rec_sel[f], "original": orig_sel[f],
                     "abs_diff": diff, "passed": diff < 1e-4})
    return rows


all_rows = []
for mark, (fname, sub) in GATE_FILES.items():
    recomputed_path = OUT[sub] / fname
    original_path = MARK1_DIR / sub / fname
    assert original_path.is_file(), f"Missing original gate: {original_path}"
    all_rows.extend(compare_gate(mark, recomputed_path, original_path))

verify_frame = pd.DataFrame(all_rows)
verify_frame.to_csv(CONSOLIDATED / "reproduction_verification.csv", index=False)
passed_count = int(verify_frame["passed"].sum())
total_count = len(verify_frame)
print(f"Reproduction verification: {passed_count}/{total_count} metric comparisons passed.")
display(verify_frame)

report = {
    "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "marks_verified": sorted(verify_frame["mark"].unique().tolist()),
    "comparisons_total": total_count,
    "comparisons_passed": passed_count,
    "all_passed": passed_count == total_count,
    "worst_abs_diff": float(verify_frame["abs_diff"].max()),
    "failures": verify_frame.loc[~verify_frame["passed"]].to_dict(orient="records"),
}
(CONSOLIDATED / "reproduction_verification.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
assert report["all_passed"], "Reproduction verification FAILED — inspect failures above."
print("PASS: unified notebook reproduces every original gate result.")

### 9.3 Consolidated findings and interpretation contract

In [ ]:
import matplotlib.patches as mpatches

# Pipeline status strip: one tile per mark
figure, axis = plt.subplots(figsize=(18, 3))
axis.set_xlim(0, 16); axis.set_ylim(0, 3); axis.axis("off")
marks = [
    ("Mark 1", "diagnostic", "FAIL (calibration)"),
    ("Mark 2", "ROI feasibility", "PASS"),
    ("Mark 3", "overfit gate", "PASS"),
    ("Mark 4", "smoke test", "5/6 targets"),
    ("Mark 4B", "calibration", "5/6 targets"),
    ("Mark 4C", "ablation", "no arm passed"),
    ("Mark 4D", "reconciliation", "V116 failure"),
    ("Mark 4E", "fusion", "PASS (all 6)"),
]
colors = ["#D66", "#6D6", "#6D6", "#DD6", "#DD6", "#D66", "#D66", "#6D6"]
for i, ((name, what, verdict), color) in enumerate(zip(marks, colors)):
    x = i * 2
    axis.add_patch(plt.Rectangle((x, 0.4), 1.85, 2.0, facecolor=color,
                                 edgecolor="#333", linewidth=1.2))
    axis.text(x + 0.925, 2.05, name, ha="center", va="center", fontsize=11, weight="bold")
    axis.text(x + 0.925, 1.45, what, ha="center", va="center", fontsize=8)
    axis.text(x + 0.925, 0.85, verdict, ha="center", va="center", fontsize=9)
    if i < len(marks) - 1:
        axis.annotate("", xy=(x + 1.9, 1.4), xytext=(x + 1.85, 1.4),
                      arrowprops={"arrowstyle": "->", "linewidth": 1.4})
axis.text(8.0, 2.6, "Mark 1 → Mark 4E research pipeline — unified reproduction",
          ha="center", fontsize=15, weight="bold")
figure.tight_layout()
figure.savefig(CONSOLIDATED / "pipeline_status_strip.png", dpi=170, bbox_inches="tight")
plt.show()

print('''
Consolidated interpretation
---------------------------
1. Mark 1 diagnosed suppressed/mis-localized tumor probability (calibration cannot recover it).
2. Mark 2 proved a prediction-only liver ROI contains 100% of every validation tumor (42.7% median crop).
3. Mark 3 proved the frozen ROI geometry + broad window can overfit (Dice 0.9006, 1-channel).
4. Mark 4 smoke reached 5/6 continuation targets; the positive predicted-empty rate stayed high.
5. Mark 4B proved threshold calibration alone cannot fix the recall failure.
6. Mark 4C showed two-channel input collapses and recall-loss alone damages V116.
7. Mark 4D reconciled metrics over the same nine positive patients and isolated V116 as a
   localization failure (median truth-region probability ~ 0), not ROI clipping.
8. Mark 4E: pixelwise maximum fusion of control + recall-loss at threshold 0.70 passes all six
   temporary validation targets -> the controlling inference policy.

Boundaries
----------
- All gates are VALIDATION gates. The test split remains locked until the complete inference policy
  is frozen and the final validation gate passes.
- Temporary continuation targets do not replace final validation targets.
- Threshold selection, fusion weights and checkpoint pairs must stay   frozen before test access.
''')

## Final takeaways

- **One notebook, eight marks.** The full Mark 1 → Mark 4E research chain now runs as a single
  structured notebook and writes every artifact (gate JSON, CSVs, figures) into
  `Evaluation/mark_1_to_4e_outputs/` with the same naming as the original `mark 1/` outputs.
- **Reproduction verified.** Part 9 compares every recomputed gate against the original gate JSONs
  and asserts all metrics agree to <1e-4 — the unified pipeline is a faithful consolidation.
- **Controlling policy.** `P_fused = max(P_control, P_recall_loss)` at threshold 0.70
  (mean positive-patient Dice 0.3771, Q1 detection 50.57%, positive empty 27.45%, empty FP 5.55%).
- **Next step** (documented in the frozen policy): bounded confirmation of the fused policy, final
  inference-policy freeze, then the one-time locked test evaluation.